# DIMER Table Intelligence Workshop
## Table Detection → Structure Recognition → Table Reconstruction → TAPAS QA

**Notebook profile:** `TASK-INFERENCE`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Workflow scope:** `COMPOSED-PIPELINE`  
**Standalone:** yes  
**Canonical workflow:** frozen inference only

This notebook treats Table Intelligence as a system: three frozen models joined by notebook-local composition rules.

### Who this notebook is for

Learners who can run cells in Colab or Jupyter and read short Python functions, and who are new to composed document-intelligence pipelines. No prior experience with Table Transformer or TAPAS is assumed. A Tesla T4-class GPU is recommended; the notebook also runs on CPU, more slowly (TAPAS Large dominates the run time).

### How to use this notebook

1. Select a T4 GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU*).
2. Choose *Runtime → Run all*. The defaults in the Configuration form (§3) are the reference settings; change a setting only in the activity marked **Predict → Change one thing** (after §18).
3. While the notebook runs, read the concept cells and write down your prediction whenever a cell asks for one, before its output appears.
4. Each section is labelled by role:
   - **Core concept** — what the models do and how the stages fit together;
   - **Evaluation practice** — how each stage is measured and how to read the numbers;
   - **Infrastructure** — runtime setup, pinned downloads, integrity checks, data plumbing and exports. You may run these cells without studying their implementation; their code is collapsed where your notebook viewer supports it.
5. Checkpoint questions have a collapsed **Sample answer**. Answer first, then open it.

### Task at a glance

```text
Input:  a document page containing one scientific table + a natural-language question
  → Table Transformer Detection       where is the table on the page?
  → crop
  → Table Transformer Structure       where are its rows and columns?
  → grid reconstruction + cell text   which string belongs in which cell?
  → TAPAS Large WTQ                   which cells answer the question, with which aggregation?
Output: an answer (cell strings or a number) + per-stage metrics showing where errors entered
```

Models:

1. **Table Transformer Detection**
2. **Table Transformer Structure Recognition v1.1-all**
3. **TAPAS Large WTQ**

The canonical corpus is the public-domain **SciTSR-PD** scientific-table dataset. Its logical cell text is used as a controlled text provider so this notebook can isolate table geometry and QA without mixing in OCR quality.

### Roadmap

1. Distinguish the three table-intelligence tasks and predict where errors will enter (§1–2).
2. Set up the runtime, models and SciTSR-PD inputs (§3–10, mostly Infrastructure).
3. Run table detection and measure it (§11–13).
4. Recognize structure on gold and detected crops, reconstruct grids and measure them (§14–18); then change the structure threshold on validation tables (activity).
5. Probe tables with spanning cells (§19).
6. Ask 50 questions through three paths with TAPAS and score them (§20–22).
7. Read the error waterfall and trace failures to their stage (§23–24).
8. Inspect resources, panels and exports (§25–28).
9. Write an evidence-based conclusion (§29); troubleshooting and a glossary follow.

### Learning objectives

By the end you should be able to:

- **distinguish** table detection, structure recognition and table QA, naming each one's input and output;
- **reconstruct** a rectangular cell grid from predicted row and column boxes;
- **trace** a box through the page → crop → table coordinate frames;
- **explain** why TAPAS needs cell strings rather than boxes, and where OCR would supply them in production;
- **compare** gold-table, structure-only and end-to-end QA accuracy on the same questions;
- **diagnose** a wrong answer by locating the stage in the error waterfall where it first went wrong;
- **predict, then test,** how one structure-threshold change moves predicted row and column counts.

> **Important boundary:** Table Transformer models recover geometry, not text. The canonical workflow uses SciTSR annotation text, not OCR inference. OCR / Document Extraction is the next notebook in the Document Intelligence track.

**AI Use Disclosure:** Generative AI assisted with this notebook’s code and instructional content under maintainer direction. The maintainer remains responsible for review, validation, and release decisions. AI-generated material may contain errors; validation claims are limited to documented runs and configurations. AI use does not imply independent verification, provider endorsement, or release approval.

## 1. Three different tasks

> **Core concept**

### Table detection
**Where is the table on the page?**

### Structure recognition
**Where are rows, columns and spanning cells inside the table?**

### Table QA
**What does the reconstructed table say?**

A correct final answer depends on all upstream interfaces.

## 2. Error waterfall

> **Core concept**

The same TAPAS questions are evaluated through three paths:

```text
Path 1:
gold logical table → TAPAS

Path 2:
gold table crop → predicted structure → reconstructed table → TAPAS

Path 3:
page → predicted table crop → predicted structure → reconstructed table → TAPAS
```

This separates QA-model error from structure error and table-detection error.

### Before you run: make predictions

Write down short answers now; §24 returns to them with the measured waterfall and sample answers.

1. Which stage is most likely to reduce end-to-end accuracy?
2. Can perfect table detection guarantee correct structure?
3. Can perfect structure guarantee correct QA?
4. Why does TAPAS need strings rather than row/column boxes?
5. Where would OCR enter a production version of this workflow?
6. What happens if one predicted row is missing?
7. What happens if the first reconstructed row is mistaken?

## 3. Configuration

> **Configuration** — the form below holds the reference settings. Keep them for `Run all`; the activity after §18 is the place to experiment.

The defaults define the canonical `Run all` path.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ""  # @param {type:"string"}

DETECTION_THRESHOLD = 0.90  # @param {type:"number"}
STRUCTURE_THRESHOLD = 0.50  # @param {type:"number"}
CROP_PADDING = 10  # @param {type:"integer"}
GRID_NMS_IOU = 0.50  # @param {type:"number"}

END_TO_END_TABLES = 10  # @param {type:"integer"}
QUESTIONS_PER_TABLE = 5

RUN_COMPLEX_STRUCTURE_PROBE = True  # @param {type:"boolean"}
COMPLEX_STRUCTURE_TABLES = 3  # @param {type:"integer"}

OUTPUT_DIR = "outputs/table_intelligence"

if not 0 <= DETECTION_THRESHOLD <= 1:
    raise ValueError("DETECTION_THRESHOLD must be in [0,1]")
if not 0 <= STRUCTURE_THRESHOLD <= 1:
    raise ValueError("STRUCTURE_THRESHOLD must be in [0,1]")
if END_TO_END_TABLES != 10:
    raise ValueError("Canonical workflow uses exactly 10 tables")
print({
    "detection_threshold":DETECTION_THRESHOLD,
    "structure_threshold":STRUCTURE_THRESHOLD,
    "crop_padding":CROP_PADDING,
    "tables":END_TO_END_TABLES,
})

## 4. Runtime

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

The three live DIMER carriers share the same Python/PyTorch runtime family. `datasets` and `pyarrow` are used only to read the small pinned SciTSR-PD parquet files.

A Tesla T4 is recommended because TAPAS Large is approximately 1.35 GB.

In [ ]:
import importlib.metadata as importlib_metadata
import subprocess, sys

PINS = {
    "torch":"2.14.0",
    "torchvision":"0.29.0",
    "torchaudio":"2.11.0",
    "transformers":"4.57.6",
    # The detection checkpoint builds its ResNet-18 backbone through timm.
    "timm":"1.0.30",
    "safetensors":"0.8.0",
    "numpy":"2.5.3",
    "pillow":"11.3.0",
    "huggingface-hub":"0.36.2",
    "datasets":"4.1.1",
    "pyarrow":"25.0.1",
    "pandas":"2.2.3",
}

def version(name):
    """The public version (PEP 440 without a local label such as +cu126), as pip compares `==` pins."""
    try:
        return importlib_metadata.version(name).split("+", 1)[0]
    except importlib_metadata.PackageNotFoundError:
        return None

# Hosted kernels such as Colab import NumPy at startup. Replacing a loaded module on disk would force a manual
# restart (Notebook Spec RUN10), so a NumPy 2.x that is already loaded is kept and recorded instead of reinstalled.
NUMPY_PRELOADED=None
if "numpy" in sys.modules and str(getattr(sys.modules["numpy"],"__version__","")).startswith("2."):
    NUMPY_PRELOADED=sys.modules["numpy"].__version__
    PINS["numpy"]=NUMPY_PRELOADED

before={k:version(k) for k in PINS}
needed=[f"{k}=={v}" for k,v in PINS.items() if before[k]!=v]
if needed:
    completed=subprocess.run(
        [sys.executable,"-m","pip","install","--quiet",*needed],
        check=False,text=True,capture_output=True,
    )
    if completed.returncode!=0:
        raise RuntimeError(
            f"pip install failed with exit code {completed.returncode}.\n"
            f"--- stderr (tail) ---\n{completed.stderr[-4000:]}\n"
            f"--- stdout (tail) ---\n{completed.stdout[-2000:]}"
        )

after={k:version(k) for k in PINS}
bad={k:(after[k],v) for k,v in PINS.items() if after[k]!=v}
if bad:
    raise RuntimeError(f"Pinned install failed: {bad}")

# Fail closed rather than run with a module whose files were replaced underneath it.
stale=[(m,sys.modules[m].__version__,after[d]) for m,d in (("numpy","numpy"),("torch","torch"),("transformers","transformers"))
       if m in sys.modules and not str(sys.modules[m].__version__).startswith(str(after[d]))]
if stale:
    raise RuntimeError(
        "The kernel had already imported packages that the pinned install replaced on disk. "
        "Restart the Python session and choose Run all again (Colab: Runtime > Restart session; "
        f"do not delete the runtime, which discards the installed pins). Stale modules: {stale}"
    )

import numpy as np
import pandas as pd
import torch
import torchvision
import transformers
import datasets
import pyarrow as pa
from PIL import Image, ImageDraw, ImageFont, ImageOps

DEVICE="cuda:0" if torch.cuda.is_available() else "cpu"
RUNTIME={
    "python":sys.version.split()[0],
    "torch":torch.__version__,
    "torchvision":torchvision.__version__,
    "transformers":transformers.__version__,
    "timm":importlib_metadata.version("timm"),
    "datasets":datasets.__version__,
    "numpy":np.__version__,
    "numpy_source":"preloaded by the host kernel" if NUMPY_PRELOADED else "pinned install",
    "pandas":pd.__version__,
    "pillow":importlib_metadata.version("pillow"),
    "device":DEVICE,
}
if torch.cuda.is_available():
    RUNTIME["gpu_name"]=torch.cuda.get_device_name(0)
print(RUNTIME)

## 5. Immutable model provenance

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

Every model is staged only from its exact immutable revision and verified against the DIMER manifest before loading.

No upstream `.bin` file is used.

In [ ]:
import hashlib, json, time, gc, math, random, re
from pathlib import Path
from huggingface_hub import hf_hub_download

DETECTION_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "table-transformer-detection", "modelId": "microsoft/table-transformer-detection", "revision": "2357cbe2b5a5d1c03e54f32764f06058933b65ab", "files": [{"path": "README.md", "bytes": 1174, "sha256": "c91e7f8199313c4d24b09e73a2f6df3141268666969346cb7b94ccb59900f5dd"}, {"path": "config.json", "bytes": 1228, "sha256": "ed5b93df2c3a59d473ddea853553a6d545d52bd4e9f8f72bf40b8a974aba4c1d"}, {"path": "model.safetensors", "bytes": 115317516, "sha256": "8f1aa73170102c038d40155e2734b343bf07e0fe12594228a8590943b01dccf7"}, {"path": "preprocessor_config.json", "bytes": 273, "sha256": "86a8837ae440456b0a9aef788b064921df29c20f0b67040954ce5c2fbd352c4f"}], "totalBytes": 115320191}""")
STRUCTURE_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "table-transformer-structure-v1.1-all", "modelId": "microsoft/table-transformer-structure-recognition-v1.1-all", "revision": "7587a7ef111d9dcbf8ac695f1376ab7014340a0c", "files": [{"path": "README.md", "bytes": 1056, "sha256": "01eed360e0f54ad297ab347cfef208d68d8ac2c108b808147bb7af9f8d9132c8"}, {"path": "config.json", "bytes": 76761, "sha256": "17a8a6edfb9e394263fa6ba9b82176ebccdfcc5d6cd29121ec91572c7d6be22c"}, {"path": "model.safetensors", "bytes": 115437156, "sha256": "9df416575a3a36ebd0129342d4f597f14d6e5170268f3d52d28584ab4466a501"}, {"path": "preprocessor_config.json", "bytes": 374, "sha256": "eead409bb80e36ae85b8377642c54550f0504f65688ba3a4967950cafe461df2"}], "totalBytes": 115515347}""")
TAPAS_MANIFEST=json.loads(r"""{"format": "dimer_hf_snapshot", "formatVersion": 1, "modelKey": "tapas-large-wtq", "modelId": "google/tapas-large-finetuned-wtq", "revision": "f58317ab2577d17647d9acafa790c744a0388b30", "files": [{"path": "README.md", "bytes": 7224, "sha256": "2397bdae46316684903465e6f425a9fe10c85d491a9755ec7a31a74353c034c8"}, {"path": "config.json", "bytes": 1659, "sha256": "834f44a325a349d2d209ecb18e4fa2ca3cdb16cef5b169e5c0a8f5e16c444d13"}, {"path": "model.safetensors", "bytes": 1346985282, "sha256": "149247e13732c222ba621e0c4e7b90ba260869b36adcbe115dc872c2c98bccf0"}, {"path": "special_tokens_map.json", "bytes": 154, "sha256": "e3ec7abc6bcd45aba696cb95e9945186dad920aea78733f8b45c83d696ed0dea"}, {"path": "tokenizer_config.json", "bytes": 490, "sha256": "ab033df4f3c902cbc66bae2736bc99f4a59c43e7d9e53e40046576826b2cc5e9"}, {"path": "vocab.txt", "bytes": 262028, "sha256": "4d96f9308bcf9019684fcc109aa8c042b9b745edabba0162fbe66c75ebee2db4"}], "totalBytes": 1347256837}""")

DETECTION_DIR=Path("weights/table-transformer-detection")
STRUCTURE_DIR=Path("weights/table-transformer-structure-v1.1-all")
TAPAS_DIR=Path("weights/tapas-large-wtq")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""):
            h.update(chunk)
    return h.hexdigest()

def stage_snapshot(root,manifest):
    root.mkdir(parents=True,exist_ok=True)
    (root/"dimer-base-manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    for entry in manifest["files"]:
        path=root/entry["path"]
        if not path.is_file():
            hf_hub_download(
                repo_id=manifest["modelId"],
                filename=entry["path"],
                revision=manifest["revision"],
                local_dir=str(root),
            )
    for entry in manifest["files"]:
        path=root/entry["path"]
        if path.stat().st_size!=entry["bytes"]:
            raise RuntimeError(f"{manifest['modelId']} {entry['path']} size mismatch")
        if sha256_file(path)!=entry["sha256"]:
            raise RuntimeError(f"{manifest['modelId']} {entry['path']} SHA-256 mismatch")
    return {"model_id":manifest["modelId"],"revision":manifest["revision"],"bytes":manifest["totalBytes"]}

print(stage_snapshot(DETECTION_DIR,DETECTION_MANIFEST))
print(stage_snapshot(STRUCTURE_DIR,STRUCTURE_MANIFEST))
print(stage_snapshot(TAPAS_DIR,TAPAS_MANIFEST))

## 6. SciTSR-PD provenance

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

Pinned dataset:

`bevaya/SciTSR-pd @ dae336efa7af07d69194a510ff5568980e8ef253`

The dataset contains 108 public-domain scientific tables (89 train / 19 test) with:

- table image;
- logical cells;
- text chunks;
- paper identity;
- source license metadata.

The two source parquet files are verified by byte size and SHA-256 before they are read.

In [ ]:
SCITSR_REPO="bevaya/SciTSR-pd"
SCITSR_REVISION="dae336efa7af07d69194a510ff5568980e8ef253"
SCITSR_FILES=json.loads(r"""{"train": {"path": "data/train-00000-of-00001.parquet", "bytes": 5185141, "sha256": "d295b918777425a26fa13a5c3ba076517429682ede8c71b671a8b6115968e465", "rows": 89}, "test": {"path": "data/test-00000-of-00001.parquet", "bytes": 1066042, "sha256": "73af6317219349a08acdf1c8fee7cff21b463cdded64111b643123f7df4f8bed", "rows": 19}}""")
SCITSR_DIR=Path("weights/scitsr-pd")
SCITSR_DIR.mkdir(parents=True,exist_ok=True)

scitsr_paths={}
for split,spec in SCITSR_FILES.items():
    path=Path(hf_hub_download(
        repo_id=SCITSR_REPO,
        repo_type="dataset",
        filename=spec["path"],
        revision=SCITSR_REVISION,
        local_dir=str(SCITSR_DIR),
    ))
    if path.stat().st_size!=spec["bytes"]:
        raise RuntimeError(f"{split} SciTSR-PD shard size mismatch")
    digest=sha256_file(path)
    if digest!=spec["sha256"]:
        raise RuntimeError(f"{split} SciTSR-PD shard SHA-256 mismatch")
    scitsr_paths[split]=path
    print(split,path,digest)

## 7. Build the table-geometry/text companion from SciTSR-PD

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

The source dataset stores logical cells (with their text) and PDF-coordinate text chunks, but no pixel boxes. The notebook combines two things:

- **Pinned carrier geometry.** The DIMER structure carrier derived pixel-space `table` / `table row` / `table column` / `table spanning cell` boxes for 94 of the 108 SciTSR-PD tables, once, and recorded the whole-pixel offset that places each table's text chunks on its image. The next cell embeds that geometry and its SHA-256.
- **Annotation text.** Each logical cell's text is matched to its PDF text chunks, and the chunk boxes are placed on the image with the carrier's offset. This gives every cell a pixel box, its row/column span and its text.

This is a deterministic data transformation, not model inference and not OCR. The notebook stops if any carrier table fails to rebuild, if its pixels differ from the carrier's, or if its cell spans disagree with the carrier's row and column counts.

In [ ]:
# Infrastructure: pinned carrier geometry for the 94 SciTSR-PD tables of the DIMER structure carrier.
# Per table: paper, chunk offset (dx, dy in pixels), grid size, pixel digest, and structure boxes
# [label, x_min, y_min, x_max, y_max] with t=table, r=table row, c=table column, s=table spanning cell.
CARRIER_GEOMETRY_SHA256="d6d524c367a151ed8a41bda7eadf7679977256953888862c9d85186eeda2c66a"
CARRIER_GEOMETRY_JSON=r"""{
"1003.3684v1.1":{"paper_id":"1003.3684","dx":20,"dy":7,"n_rows":3,"n_cols":4,"pixel_sha256":"f81a6fd18dac7606a157db5a95e666000dc72017d41f82dd65d2ade9a8d47491","objects":[["t",3.0,3.0,563.0,84.0],["r",3.0,3.0,563.0,30.9],["r",3.0,30.9,563.0,58.3],["r",3.0,58.3,563.0,84.0],["c",3.0,3.0,113.1,84.0],["c",113.1,3.0,255.3,84.0],["c",255.3,3.0,391.0,84.0],["c",391.0,3.0,563.0,84.0]]},
"1003.3684v1.2":{"paper_id":"1003.3684","dx":20,"dy":15,"n_rows":5,"n_cols":3,"pixel_sha256":"e9980448fa314c2941757c1c59bd86edc8ede136bdf01c0485a922899797af92","objects":[["t",3.0,3.0,577.0,142.0],["r",3.0,3.0,577.0,35.1],["r",3.0,35.1,577.0,62.5],["r",3.0,62.5,577.0,87.4],["r",3.0,87.4,577.0,112.3],["r",3.0,112.3,577.0,142.0],["c",3.0,3.0,161.2,142.0],["c",161.2,3.0,352.2,142.0],["c",352.2,3.0,577.0,142.0]]},
"1004.5186v1.1":{"paper_id":"1004.5186","dx":16,"dy":11,"n_rows":6,"n_cols":3,"pixel_sha256":"c3766c7010adb1c83c11546bdf2fc677937e4d10d058e2141aa09e8c868bbc02","objects":[["t",3.0,3.0,426.0,155.0],["r",3.0,3.0,426.0,29.1],["r",3.0,29.1,426.0,54.4],["r",3.0,54.4,426.0,79.3],["r",3.0,79.3,426.0,104.2],["r",3.0,104.2,426.0,129.1],["r",3.0,129.1,426.0,155.0],["c",3.0,3.0,184.2,155.0],["c",184.2,3.0,283.6,155.0],["c",283.6,3.0,426.0,155.0]]},
"1007.0920v1.1":{"paper_id":"1007.0920","dx":16,"dy":11,"n_rows":8,"n_cols":4,"pixel_sha256":"40a7a095db0e2cebb6ab5ca5745f168fb4924962f03598ebba7890ed817f0cf7","objects":[["t",3.0,3.0,543.0,213.0],["r",3.0,3.0,543.0,31.1],["r",3.0,31.1,543.0,58.5],["r",3.0,58.5,543.0,83.4],["r",3.0,83.4,543.0,108.3],["r",3.0,108.3,543.0,133.3],["r",3.0,133.3,543.0,158.2],["r",3.0,158.2,543.0,183.1],["r",3.0,183.1,543.0,213.0],["c",3.0,3.0,115.4,213.0],["c",115.4,3.0,263.6,213.0],["c",263.6,3.0,396.7,213.0],["c",396.7,3.0,543.0,213.0]]},
"1007.0920v1.2":{"paper_id":"1007.0920","dx":16,"dy":12,"n_rows":13,"n_cols":4,"pixel_sha256":"175b2b08e0b134f8ca6a6b1adfa2f3555c15ec637d60ec937415d9f2aa0ca8a3","objects":[["t",3.0,3.0,549.0,338.0],["r",3.0,3.0,549.0,32.1],["r",3.0,32.1,549.0,59.5],["r",3.0,59.5,549.0,84.4],["r",3.0,84.4,549.0,109.3],["r",3.0,109.3,549.0,134.3],["r",3.0,134.3,549.0,159.2],["r",3.0,159.2,549.0,184.1],["r",3.0,184.1,549.0,209.0],["r",3.0,209.0,549.0,233.9],["r",3.0,233.9,549.0,258.8],["r",3.0,258.8,549.0,283.7],["r",3.0,283.7,549.0,308.6],["r",3.0,308.6,549.0,338.0],["c",3.0,3.0,118.4,338.0],["c",118.4,3.0,266.5,338.0],["c",266.5,3.0,402.7,338.0],["c",402.7,3.0,549.0,338.0]]},
"1108.4723v1.1":{"paper_id":"1108.4723","dx":16,"dy":12,"n_rows":5,"n_cols":7,"pixel_sha256":"085f89838b3a7b2dd6765406ac806e07217ccfce3dcfb221b207efe93c6df0cd","objects":[["t",3.0,3.0,641.0,137.0],["r",3.0,3.0,641.0,32.1],["r",3.0,32.1,641.0,59.9],["r",3.0,59.9,641.0,87.2],["r",3.0,87.2,641.0,113.0],["r",3.0,113.0,641.0,137.0],["c",3.0,3.0,138.7,137.0],["c",138.7,3.0,224.1,137.0],["c",224.1,3.0,307.4,137.0],["c",307.4,3.0,390.7,137.0],["c",390.7,3.0,474.0,137.0],["c",474.0,3.0,557.3,137.0],["c",557.3,3.0,641.0,137.0],["s",474.0,3.0,641.0,32.1],["s",138.7,3.0,307.4,32.1],["s",307.4,3.0,474.0,32.1]]},
"1108.4723v1.2":{"paper_id":"1108.4723","dx":16,"dy":11,"n_rows":6,"n_cols":10,"pixel_sha256":"6decf00a18f5937e7f01a59de2a6504b5ef515bcf870f1a1d5ea79294aadadb8","objects":[["t",3.0,3.0,706.0,162.0],["r",3.0,3.0,706.0,29.1],["r",3.0,29.1,706.0,56.9],["r",3.0,56.9,706.0,84.7],["r",3.0,84.7,706.0,110.4],["r",3.0,110.4,706.0,136.2],["r",3.0,136.2,706.0,162.0],["c",3.0,3.0,138.7,162.0],["c",138.7,3.0,195.4,162.0],["c",195.4,3.0,262.1,162.0],["c",262.1,3.0,328.2,162.0],["c",328.2,3.0,384.2,162.0],["c",384.2,3.0,450.9,162.0],["c",450.9,3.0,517.1,162.0],["c",517.1,3.0,573.1,162.0],["c",573.1,3.0,639.8,162.0],["c",639.8,3.0,706.0,162.0],["s",328.2,3.0,517.1,29.1],["s",138.7,3.0,328.2,29.1],["s",517.1,3.0,706.0,29.1]]},
"1108.4723v1.3":{"paper_id":"1108.4723","dx":16,"dy":12,"n_rows":4,"n_cols":4,"pixel_sha256":"668cb76ec6d117e60025839f730cb21c8e1648909d357792571066d98e5884f4","objects":[["t",3.0,3.0,395.0,111.0],["r",3.0,3.0,395.0,32.1],["r",3.0,32.1,395.0,59.9],["r",3.0,59.9,395.0,85.7],["r",3.0,85.7,395.0,111.0],["c",3.0,3.0,113.6,111.0],["c",113.6,3.0,208.6,111.0],["c",208.6,3.0,301.6,111.0],["c",301.6,3.0,395.0,111.0]]},
"1108.4723v1.4":{"paper_id":"1108.4723","dx":16,"dy":11,"n_rows":3,"n_cols":4,"pixel_sha256":"38801d9a4ce81bcf62b0cf301a6a6e79182fa8b019e1605757bf14cee0ebd2fa","objects":[["t",3.0,3.0,374.0,86.0],["r",3.0,3.0,374.0,31.1],["r",3.0,31.1,374.0,58.9],["r",3.0,58.9,374.0,86.0],["c",3.0,3.0,92.8,86.0],["c",92.8,3.0,187.8,86.0],["c",187.8,3.0,280.7,86.0],["c",280.7,3.0,374.0,86.0]]},
"1109.4653v2.11":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"d4ce0897e5e92fc558df075300c7ec8b1f944eaf541fdd1df7967282f36fba4f","objects":[["t",3.0,3.0,401.0,184.0],["r",3.0,3.0,401.0,30.8],["r",3.0,30.8,401.0,56.6],["r",3.0,56.6,401.0,81.5],["r",3.0,81.5,401.0,106.4],["r",3.0,106.4,401.0,131.4],["r",3.0,131.4,401.0,156.3],["r",3.0,156.3,401.0,184.0],["c",3.0,3.0,232.7,184.0],["c",232.7,3.0,317.4,184.0],["c",317.4,3.0,401.0,184.0]]},
"1109.4653v2.12":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":6,"n_cols":2,"pixel_sha256":"172f29e8a1e10efb4d5e27498a8e920f399b3f3665458f482d08e35dcbdca1cb","objects":[["t",3.0,3.0,432.0,159.0],["r",3.0,3.0,432.0,30.8],["r",3.0,30.8,432.0,56.6],["r",3.0,56.6,432.0,81.5],["r",3.0,81.5,432.0,106.4],["r",3.0,106.4,432.0,131.4],["r",3.0,131.4,432.0,159.0],["c",3.0,3.0,346.5,159.0],["c",346.5,3.0,432.0,159.0]]},
"1109.4653v2.2":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":8,"n_cols":2,"pixel_sha256":"74ce1641c8ef8a4593a612a484893e9657af34133390f9d4edb76d863e4236d2","objects":[["t",3.0,3.0,371.0,205.0],["r",3.0,3.0,371.0,29.1],["r",3.0,29.1,371.0,54.4],["r",3.0,54.4,371.0,79.3],["r",3.0,79.3,371.0,102.3],["r",3.0,102.3,371.0,129.1],["r",3.0,129.1,371.0,154.0],["r",3.0,154.0,371.0,178.9],["r",3.0,178.9,371.0,205.0],["c",3.0,3.0,143.1,205.0],["c",143.1,3.0,371.0,205.0]]},
"1109.4653v2.6":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":17,"n_cols":3,"pixel_sha256":"9a6c61d297851c0f0545b71b76ce6ed8cd3d7d50a8f93240c34691b409e2ddb3","objects":[["t",3.0,3.0,318.0,435.0],["r",3.0,3.0,318.0,30.6],["r",3.0,30.6,318.0,54.4],["r",3.0,54.4,318.0,79.3],["r",3.0,79.3,318.0,104.2],["r",3.0,104.2,318.0,129.1],["r",3.0,129.1,318.0,154.0],["r",3.0,154.0,318.0,178.9],["r",3.0,178.9,318.0,206.3],["r",3.0,206.3,318.0,235.7],["r",3.0,235.7,318.0,261.0],["r",3.0,261.0,318.0,285.9],["r",3.0,285.9,318.0,310.8],["r",3.0,310.8,318.0,335.7],["r",3.0,335.7,318.0,360.6],["r",3.0,360.6,318.0,385.5],["r",3.0,385.5,318.0,410.4],["r",3.0,410.4,318.0,435.0],["c",3.0,3.0,143.1,435.0],["c",143.1,3.0,234.6,435.0],["c",234.6,3.0,318.0,435.0]]},
"1109.4653v2.8":{"paper_id":"1109.4653","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"78a47c83e462d7e89825dd107ba341471cbc91fda9ab9ce878f033f51f4eec26","objects":[["t",3.0,3.0,441.0,184.0],["r",3.0,3.0,441.0,30.8],["r",3.0,30.8,441.0,56.6],["r",3.0,56.6,441.0,81.5],["r",3.0,81.5,441.0,106.4],["r",3.0,106.4,441.0,131.4],["r",3.0,131.4,441.0,156.3],["r",3.0,156.3,441.0,184.0],["c",3.0,3.0,273.5,184.0],["c",273.5,3.0,358.2,184.0],["c",358.2,3.0,441.0,184.0]]},
"1109.4653v2.9":{"paper_id":"1109.4653","dx":17,"dy":11,"n_rows":6,"n_cols":2,"pixel_sha256":"e375479a3eeaff2e3b4a66006fdda11fc781b2d489d69832e0d5c1b7f8f35686","objects":[["t",3.0,3.0,464.0,159.0],["r",3.0,3.0,464.0,30.8],["r",3.0,30.8,464.0,56.6],["r",3.0,56.6,464.0,81.5],["r",3.0,81.5,464.0,106.4],["r",3.0,106.4,464.0,131.4],["r",3.0,131.4,464.0,159.0],["c",3.0,3.0,379.4,159.0],["c",379.4,3.0,464.0,159.0]]},
"1301.0302v2.1":{"paper_id":"1301.0302","dx":17,"dy":11,"n_rows":8,"n_cols":6,"pixel_sha256":"9b1bb9f3d19c50a7a850dcd9f78bca5fcedf10e2aa46bb39091f158ce2cc0cf1","objects":[["t",3.0,3.0,958.0,214.0],["r",3.0,3.0,958.0,31.1],["r",3.0,31.1,958.0,58.9],["r",3.0,58.9,958.0,84.7],["r",3.0,84.7,958.0,110.4],["r",3.0,110.4,958.0,136.2],["r",3.0,136.2,958.0,161.9],["r",3.0,161.9,958.0,187.6],["r",3.0,187.6,958.0,214.0],["c",3.0,3.0,415.2,214.0],["c",415.2,3.0,549.4,214.0],["c",549.4,3.0,654.2,214.0],["c",654.2,3.0,748.9,214.0],["c",748.9,3.0,843.5,214.0],["c",843.5,3.0,958.0,214.0]]},
"1305.1199v4.1":{"paper_id":"1305.1199","dx":23,"dy":16,"n_rows":7,"n_cols":2,"pixel_sha256":"9a3b144c34d0a82d6bb55918ce1244bd6b3f5526ff9c09e2b3e6df32536f4da5","objects":[["t",3.0,3.0,962.0,228.0],["r",3.0,3.0,962.0,37.2],["r",3.0,37.2,962.0,69.4],["r",3.0,69.4,962.0,101.5],["r",3.0,101.5,962.0,133.6],["r",3.0,133.6,962.0,165.7],["r",3.0,165.7,962.0,197.8],["r",3.0,197.8,962.0,228.0],["c",3.0,3.0,553.1,228.0],["c",553.1,3.0,962.0,228.0]]},
"1401.1475v1.1":{"paper_id":"1401.1475","dx":16,"dy":9,"n_rows":7,"n_cols":2,"pixel_sha256":"2a545f13c0112038304e5f3436ecea06257ea9cb41e4a04512429eca2e65986b","objects":[["t",3.0,3.0,699.0,149.0],["r",3.0,3.0,699.0,25.5],["r",3.0,25.5,699.0,47.7],["r",3.0,47.7,699.0,67.8],["r",3.0,67.8,699.0,88.0],["r",3.0,88.0,699.0,108.1],["r",3.0,108.1,699.0,128.2],["r",3.0,128.2,699.0,149.0],["c",3.0,3.0,352.9,149.0],["c",352.9,3.0,699.0,149.0]]},
"1404.5002v1.1":{"paper_id":"1404.5002","dx":16,"dy":10,"n_rows":13,"n_cols":5,"pixel_sha256":"72c4ddb559b2468d4888c4922bfed0b0eef18929bb2ce17553de07d22c59fc49","objects":[["t",3.0,3.0,488.0,263.0],["r",3.0,3.0,488.0,24.0],["r",3.0,24.0,488.0,44.1],["r",3.0,44.1,488.0,64.3],["r",3.0,64.3,488.0,84.0],["r",3.0,84.0,488.0,103.7],["r",3.0,103.7,488.0,123.4],["r",3.0,123.4,488.0,143.1],["r",3.0,143.1,488.0,162.9],["r",3.0,162.9,488.0,182.6],["r",3.0,182.6,488.0,202.3],["r",3.0,202.3,488.0,222.4],["r",3.0,222.4,488.0,242.6],["r",3.0,242.6,488.0,263.0],["c",3.0,3.0,133.9,263.0],["c",133.9,3.0,220.7,263.0],["c",220.7,3.0,305.7,263.0],["c",305.7,3.0,366.8,263.0],["c",366.8,3.0,488.0,263.0]]},
"1404.5002v1.2":{"paper_id":"1404.5002","dx":16,"dy":9,"n_rows":12,"n_cols":5,"pixel_sha256":"2b78851e895d47604289323e6ca1c439ca8b9438b132a14ef9ca755b7727fa11","objects":[["t",3.0,3.0,453.0,243.0],["r",3.0,3.0,453.0,23.4],["r",3.0,23.4,453.0,43.6],["r",3.0,43.6,453.0,63.3],["r",3.0,63.3,453.0,83.0],["r",3.0,83.0,453.0,102.7],["r",3.0,102.7,453.0,122.4],["r",3.0,122.4,453.0,142.1],["r",3.0,142.1,453.0,161.9],["r",3.0,161.9,453.0,181.6],["r",3.0,181.6,453.0,201.7],["r",3.0,201.7,453.0,221.8],["r",3.0,221.8,453.0,243.0],["c",3.0,3.0,133.9,243.0],["c",133.9,3.0,251.1,243.0],["c",251.1,3.0,319.9,243.0],["c",319.9,3.0,385.5,243.0],["c",385.5,3.0,453.0,243.0]]},
"1407.3745v1.1":{"paper_id":"1407.3745","dx":16,"dy":15,"n_rows":4,"n_cols":4,"pixel_sha256":"2fafa59648427ee3b5cf512202acb49df2d0d29892abd6ef324fd7fad7af2c0d","objects":[["t",3.0,3.0,717.0,111.0],["r",3.0,3.0,717.0,33.1],["r",3.0,33.1,717.0,58.8],["r",3.0,58.8,717.0,84.5],["r",3.0,84.5,717.0,111.0],["c",3.0,3.0,326.6,111.0],["c",326.6,3.0,487.7,111.0],["c",487.7,3.0,596.8,111.0],["c",596.8,3.0,717.0,111.0]]},
"1409.5313v2.2":{"paper_id":"1409.5313","dx":16,"dy":10,"n_rows":7,"n_cols":5,"pixel_sha256":"cb07864095b3591ea250d54ec109a836aee2937118eb87a094f166c8f1060877","objects":[["t",3.0,3.0,587.0,148.0],["r",3.0,3.0,587.0,24.4],["r",3.0,24.4,587.0,45.0],["r",3.0,45.0,587.0,65.5],["r",3.0,65.5,587.0,86.1],["r",3.0,86.1,587.0,106.6],["r",3.0,106.6,587.0,127.2],["r",3.0,127.2,587.0,148.0],["c",3.0,3.0,129.3,148.0],["c",129.3,3.0,248.0,148.0],["c",248.0,3.0,354.6,148.0],["c",354.6,3.0,457.2,148.0],["c",457.2,3.0,587.0,148.0]]},
"1410.1237v2.1":{"paper_id":"1410.1237","dx":16,"dy":11,"n_rows":13,"n_cols":6,"pixel_sha256":"9b0b863353c18d2813c92258ad395841c746e6f30aff0216ea3ded5dfaff738f","objects":[["t",3.0,3.0,743.0,346.0],["r",3.0,3.0,743.0,28.6],["r",3.0,28.6,743.0,56.0],["r",3.0,56.0,743.0,83.9],["r",3.0,83.9,743.0,109.6],["r",3.0,109.6,743.0,135.3],["r",3.0,135.3,743.0,161.1],["r",3.0,161.1,743.0,186.8],["r",3.0,186.8,743.0,212.5],["r",3.0,212.5,743.0,238.3],["r",3.0,238.3,743.0,264.0],["r",3.0,264.0,743.0,289.7],["r",3.0,289.7,743.0,315.5],["r",3.0,315.5,743.0,346.0],["c",3.0,3.0,184.2,346.0],["c",184.2,3.0,311.9,346.0],["c",311.9,3.0,457.9,346.0],["c",457.9,3.0,566.9,346.0],["c",566.9,3.0,659.9,346.0],["c",659.9,3.0,743.0,346.0],["s",457.9,3.0,743.0,28.6]]},
"1510.01234v1.1":{"paper_id":"1510.01234","dx":16,"dy":11,"n_rows":13,"n_cols":2,"pixel_sha256":"e27955a634e3bf33e8aab42ba14d0f79464b31e1edd3b408f2d5e74dc5f60570","objects":[["t",3.0,3.0,365.0,329.0],["r",3.0,3.0,365.0,29.1],["r",3.0,29.1,365.0,54.4],["r",3.0,54.4,365.0,79.3],["r",3.0,79.3,365.0,104.2],["r",3.0,104.2,365.0,129.1],["r",3.0,129.1,365.0,154.0],["r",3.0,154.0,365.0,178.9],["r",3.0,178.9,365.0,203.8],["r",3.0,203.8,365.0,228.7],["r",3.0,228.7,365.0,253.6],["r",3.0,253.6,365.0,278.5],["r",3.0,278.5,365.0,303.4],["r",3.0,303.4,365.0,329.0],["c",3.0,3.0,261.4,329.0],["c",261.4,3.0,365.0,329.0]]},
"1510.01234v1.2":{"paper_id":"1510.01234","dx":2,"dy":11,"n_rows":17,"n_cols":2,"pixel_sha256":"7f39f46a90ab914af315df25d34720c1a961b415a7fb90ac6c887e8d8a547c86","objects":[["t",3.0,3.0,390.0,443.0],["r",3.0,3.0,390.0,29.1],["r",3.0,29.1,390.0,54.4],["r",3.0,54.4,390.0,79.3],["r",3.0,79.3,390.0,104.2],["r",3.0,104.2,390.0,129.1],["r",3.0,129.1,390.0,154.0],["r",3.0,154.0,390.0,178.9],["r",3.0,178.9,390.0,203.8],["r",3.0,203.8,390.0,228.7],["r",3.0,228.7,390.0,253.6],["r",3.0,253.6,390.0,278.5],["r",3.0,278.5,390.0,303.4],["r",3.0,303.4,390.0,328.4],["r",3.0,328.4,390.0,353.3],["r",3.0,353.3,390.0,378.2],["r",3.0,378.2,390.0,403.1],["r",3.0,403.1,390.0,443.0],["c",3.0,3.0,261.4,443.0],["c",261.4,3.0,390.0,443.0]]},
"1510.04780v1.6":{"paper_id":"1510.04780","dx":16,"dy":12,"n_rows":31,"n_cols":2,"pixel_sha256":"abd6b413d0a56a3513e09a46db71ec5583a1aa541fb9a99ccaa6272b42b82ecc","objects":[["t",3.0,3.0,703.0,778.0],["r",3.0,3.0,703.0,30.1],["r",3.0,30.1,703.0,55.4],["r",3.0,55.4,703.0,80.3],["r",3.0,80.3,703.0,105.2],["r",3.0,105.2,703.0,130.1],["r",3.0,130.1,703.0,155.0],["r",3.0,155.0,703.0,179.9],["r",3.0,179.9,703.0,204.8],["r",3.0,204.8,703.0,229.7],["r",3.0,229.7,703.0,254.6],["r",3.0,254.6,703.0,279.5],["r",3.0,279.5,703.0,304.4],["r",3.0,304.4,703.0,329.4],["r",3.0,329.4,703.0,354.3],["r",3.0,354.3,703.0,379.2],["r",3.0,379.2,703.0,404.1],["r",3.0,404.1,703.0,429.0],["r",3.0,429.0,703.0,453.9],["r",3.0,453.9,703.0,478.8],["r",3.0,478.8,703.0,503.7],["r",3.0,503.7,703.0,528.6],["r",3.0,528.6,703.0,553.5],["r",3.0,553.5,703.0,578.4],["r",3.0,578.4,703.0,603.3],["r",3.0,603.3,703.0,628.2],["r",3.0,628.2,703.0,653.1],["r",3.0,653.1,703.0,678.0],["r",3.0,678.0,703.0,703.0],["r",3.0,703.0,703.0,727.9],["r",3.0,727.9,703.0,752.8],["r",3.0,752.8,703.0,778.0],["c",3.0,3.0,75.7,778.0],["c",75.7,3.0,703.0,778.0]]},
"1510.04780v1.7":{"paper_id":"1510.04780","dx":16,"dy":11,"n_rows":14,"n_cols":2,"pixel_sha256":"8973451847fdfba22d0674bd75dbf117ef16c90f88c8425eb7c6d2733c3a4b4b","objects":[["t",3.0,3.0,814.0,354.0],["r",3.0,3.0,814.0,29.1],["r",3.0,29.1,814.0,54.4],["r",3.0,54.4,814.0,79.3],["r",3.0,79.3,814.0,104.2],["r",3.0,104.2,814.0,129.1],["r",3.0,129.1,814.0,154.0],["r",3.0,154.0,814.0,178.9],["r",3.0,178.9,814.0,203.8],["r",3.0,203.8,814.0,228.7],["r",3.0,228.7,814.0,253.6],["r",3.0,253.6,814.0,278.5],["r",3.0,278.5,814.0,303.4],["r",3.0,303.4,814.0,328.4],["r",3.0,328.4,814.0,354.0],["c",3.0,3.0,65.4,354.0],["c",65.4,3.0,814.0,354.0]]},
"1511.06278v1.1":{"paper_id":"1511.06278","dx":4,"dy":11,"n_rows":6,"n_cols":12,"pixel_sha256":"6a32a72562a3d07b494415feb93a7aa19cb2d0aff6ae2e97f9259d03d1aa88c2","objects":[["t",3.0,3.0,585.0,162.0],["r",3.0,3.0,585.0,31.1],["r",3.0,31.1,585.0,58.9],["r",3.0,58.9,585.0,84.7],["r",3.0,84.7,585.0,110.4],["r",3.0,110.4,585.0,136.2],["r",3.0,136.2,585.0,162.0],["c",3.0,3.0,66.2,162.0],["c",66.2,3.0,119.3,162.0],["c",119.3,3.0,166.7,162.0],["c",166.7,3.0,212.4,162.0],["c",212.4,3.0,258.0,162.0],["c",258.0,3.0,303.7,162.0],["c",303.7,3.0,349.4,162.0],["c",349.4,3.0,395.0,162.0],["c",395.0,3.0,440.7,162.0],["c",440.7,3.0,486.3,162.0],["c",486.3,3.0,532.0,162.0],["c",532.0,3.0,585.0,162.0]]},
"1602.02332v1.3":{"paper_id":"1602.02332","dx":16,"dy":11,"n_rows":12,"n_cols":3,"pixel_sha256":"5beed8244b2508e66133342d2604d074a201da7df3558207126a65723752fd1d","objects":[["t",3.0,3.0,584.0,305.0],["r",3.0,3.0,584.0,29.1],["r",3.0,29.1,584.0,54.4],["r",3.0,54.4,584.0,79.3],["r",3.0,79.3,584.0,104.2],["r",3.0,104.2,584.0,129.1],["r",3.0,129.1,584.0,154.0],["r",3.0,154.0,584.0,178.9],["r",3.0,178.9,584.0,203.8],["r",3.0,203.8,584.0,228.7],["r",3.0,228.7,584.0,253.6],["r",3.0,253.6,584.0,278.5],["r",3.0,278.5,584.0,305.0],["c",3.0,3.0,210.7,305.0],["c",210.7,3.0,454.9,305.0],["c",454.9,3.0,584.0,305.0]]},
"1603.01595v1.1":{"paper_id":"1603.01595","dx":17,"dy":12,"n_rows":8,"n_cols":4,"pixel_sha256":"6101172539a340efbd8097c085dfc091d859b04cb2f6cb21ca6abcc1ca2ec47d","objects":[["t",3.0,3.0,508.0,205.0],["r",3.0,3.0,508.0,30.1],["r",3.0,30.1,508.0,55.4],["r",3.0,55.4,508.0,80.3],["r",3.0,80.3,508.0,105.2],["r",3.0,105.2,508.0,130.1],["r",3.0,130.1,508.0,155.0],["r",3.0,155.0,508.0,179.9],["r",3.0,179.9,508.0,205.0],["c",3.0,3.0,185.7,205.0],["c",185.7,3.0,278.3,205.0],["c",278.3,3.0,397.9,205.0],["c",397.9,3.0,508.0,205.0]]},
"1603.01595v1.2":{"paper_id":"1603.01595","dx":16,"dy":12,"n_rows":6,"n_cols":4,"pixel_sha256":"71b224d18b7b4fee54e66f3cd7a6a167cd487c387d2e23949756e03a6d1fa81f","objects":[["t",3.0,3.0,596.0,155.0],["r",3.0,3.0,596.0,30.1],["r",3.0,30.1,596.0,55.4],["r",3.0,55.4,596.0,80.3],["r",3.0,80.3,596.0,105.2],["r",3.0,105.2,596.0,130.1],["r",3.0,130.1,596.0,155.0],["c",3.0,3.0,301.0,155.0],["c",301.0,3.0,383.5,155.0],["c",383.5,3.0,490.8,155.0],["c",490.8,3.0,596.0,155.0]]},
"1603.01595v1.3":{"paper_id":"1603.01595","dx":16,"dy":11,"n_rows":12,"n_cols":2,"pixel_sha256":"7220c236a75da7df29eaf041892f949055f685423e91ed4a280fa30fbab1f055","objects":[["t",3.0,3.0,317.0,304.0],["r",3.0,3.0,317.0,29.1],["r",3.0,29.1,317.0,54.4],["r",3.0,54.4,317.0,79.3],["r",3.0,79.3,317.0,104.2],["r",3.0,104.2,317.0,129.1],["r",3.0,129.1,317.0,154.0],["r",3.0,154.0,317.0,178.9],["r",3.0,178.9,317.0,203.8],["r",3.0,203.8,317.0,228.7],["r",3.0,228.7,317.0,253.6],["r",3.0,253.6,317.0,278.5],["r",3.0,278.5,317.0,304.0],["c",3.0,3.0,151.7,304.0],["c",151.7,3.0,317.0,304.0]]},
"1603.02514v3.1":{"paper_id":"1603.02514","dx":16,"dy":11,"n_rows":3,"n_cols":5,"pixel_sha256":"3e7f4a09541fb2069e99b9ff1e58ffc56ab32aea8853d935e34408207cc22313","objects":[["t",3.0,3.0,583.0,74.0],["r",3.0,3.0,583.0,27.5],["r",3.0,27.5,583.0,50.7],["r",3.0,50.7,583.0,74.0],["c",3.0,3.0,121.1,74.0],["c",121.1,3.0,232.9,74.0],["c",232.9,3.0,369.2,74.0],["c",369.2,3.0,476.1,74.0],["c",476.1,3.0,583.0,74.0]]},
"1604.06285v1.2":{"paper_id":"1604.06285","dx":16,"dy":11,"n_rows":14,"n_cols":2,"pixel_sha256":"4821c2aa4c1da202523777ee33a0c8f64a704738a096b8434aae91aa7a3cec9a","objects":[["t",3.0,3.0,498.0,359.0],["r",3.0,3.0,498.0,29.1],["r",3.0,29.1,498.0,54.8],["r",3.0,54.8,498.0,80.1],["r",3.0,80.1,498.0,105.0],["r",3.0,105.0,498.0,129.9],["r",3.0,129.9,498.0,155.3],["r",3.0,155.3,498.0,181.0],["r",3.0,181.0,498.0,206.3],["r",3.0,206.3,498.0,231.2],["r",3.0,231.2,498.0,256.1],["r",3.0,256.1,498.0,281.4],["r",3.0,281.4,498.0,307.2],["r",3.0,307.2,498.0,330.6],["r",3.0,330.6,498.0,359.0],["c",3.0,3.0,62.4,359.0],["c",62.4,3.0,498.0,359.0],["s",3.0,29.1,498.0,54.8],["s",3.0,155.3,498.0,181.0],["s",3.0,281.4,498.0,307.2]]},
"1604.06285v1.4":{"paper_id":"1604.06285","dx":16,"dy":11,"n_rows":5,"n_cols":5,"pixel_sha256":"49a9915485b4745da5f107d9807bb5d5e537fccb6636dbbc3732de912ea4452e","objects":[["t",3.0,3.0,409.0,131.0],["r",3.0,3.0,409.0,29.1],["r",3.0,29.1,409.0,54.4],["r",3.0,54.4,409.0,79.7],["r",3.0,79.7,409.0,105.0],["r",3.0,105.0,409.0,131.0],["c",3.0,3.0,159.1,131.0],["c",159.1,3.0,222.8,131.0],["c",222.8,3.0,284.6,131.0],["c",284.6,3.0,346.4,131.0],["c",346.4,3.0,409.0,131.0],["s",3.0,79.7,159.1,131.0],["s",3.0,29.1,159.1,79.7]]},
"1604.06979v1.1":{"paper_id":"1604.06979","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"9ea947d633a7fde0fef6cc4a891d4c44b086ea63b272d8e23f72844d098d246f","objects":[["t",3.0,3.0,580.0,188.0],["r",3.0,3.0,580.0,31.1],["r",3.0,31.1,580.0,58.9],["r",3.0,58.9,580.0,84.7],["r",3.0,84.7,580.0,110.4],["r",3.0,110.4,580.0,136.2],["r",3.0,136.2,580.0,161.9],["r",3.0,161.9,580.0,188.0],["c",3.0,3.0,287.5,188.0],["c",287.5,3.0,444.8,188.0],["c",444.8,3.0,580.0,188.0]]},
"1604.06979v1.2":{"paper_id":"1604.06979","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"cdaedeba73097ba9446c521800d9c546796adc5f70ef1fa24343581d46287760","objects":[["t",3.0,3.0,476.0,188.0],["r",3.0,3.0,476.0,31.1],["r",3.0,31.1,476.0,58.9],["r",3.0,58.9,476.0,84.7],["r",3.0,84.7,476.0,110.4],["r",3.0,110.4,476.0,136.2],["r",3.0,136.2,476.0,161.9],["r",3.0,161.9,476.0,188.0],["c",3.0,3.0,190.9,188.0],["c",190.9,3.0,341.2,188.0],["c",341.2,3.0,476.0,188.0]]},
"1605.04635v2.1":{"paper_id":"1605.04635","dx":16,"dy":12,"n_rows":4,"n_cols":5,"pixel_sha256":"be421a209c4bd21f5bfdc26f83bace51de13925e35fd80e1ea38ecb4675bcc6a","objects":[["t",3.0,3.0,470.0,107.0],["r",3.0,3.0,470.0,29.0],["r",3.0,29.0,470.0,54.8],["r",3.0,54.8,470.0,80.5],["r",3.0,80.5,470.0,107.0],["c",3.0,3.0,106.6,107.0],["c",106.6,3.0,197.6,107.0],["c",197.6,3.0,280.4,107.0],["c",280.4,3.0,401.1,107.0],["c",401.1,3.0,470.0,107.0]]},
"1605.06770v1.1":{"paper_id":"1605.06770","dx":16,"dy":12,"n_rows":8,"n_cols":2,"pixel_sha256":"fef7a93d63404699c3a70726df6a5eee1c913f837cffc7cd3a5745619c9b2f87","objects":[["t",3.0,3.0,513.0,209.0],["r",3.0,3.0,513.0,32.1],["r",3.0,32.1,513.0,59.5],["r",3.0,59.5,513.0,84.4],["r",3.0,84.4,513.0,109.3],["r",3.0,109.3,513.0,134.3],["r",3.0,134.3,513.0,159.2],["r",3.0,159.2,513.0,184.1],["r",3.0,184.1,513.0,209.0],["c",3.0,3.0,419.6,209.0],["c",419.6,3.0,513.0,209.0]]},
"1605.06770v1.2":{"paper_id":"1605.06770","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"7c7e6332e382b62ce80b89ca72f56c0b413f0c9252c402904c75fe5094018b1d","objects":[["t",3.0,3.0,473.0,140.0],["r",3.0,3.0,473.0,31.1],["r",3.0,31.1,473.0,58.5],["r",3.0,58.5,473.0,87.5],["r",3.0,87.5,473.0,113.3],["r",3.0,113.3,473.0,140.0],["c",3.0,3.0,101.9,140.0],["c",101.9,3.0,226.3,140.0],["c",226.3,3.0,324.3,140.0],["c",324.3,3.0,473.0,140.0],["s",3.0,87.5,101.9,140.0],["s",3.0,31.1,101.9,87.5]]},
"1606.09370v1.1":{"paper_id":"1606.09370","dx":16,"dy":12,"n_rows":7,"n_cols":2,"pixel_sha256":"860111df9549d7448d973fd315aeadb65ff1c2d9d6de5c140a3501a4e30677d1","objects":[["t",3.0,3.0,351.0,184.0],["r",3.0,3.0,351.0,30.1],["r",3.0,30.1,351.0,55.8],["r",3.0,55.8,351.0,81.5],["r",3.0,81.5,351.0,107.3],["r",3.0,107.3,351.0,133.0],["r",3.0,133.0,351.0,158.7],["r",3.0,158.7,351.0,184.0],["c",3.0,3.0,137.7,184.0],["c",137.7,3.0,351.0,184.0]]},
"1606.09370v1.4":{"paper_id":"1606.09370","dx":16,"dy":11,"n_rows":7,"n_cols":4,"pixel_sha256":"a7cd36495249046b748241e70dcfd25286acfe2e669aa2aae18b8786efb51f48","objects":[["t",3.0,3.0,417.0,193.0],["r",3.0,3.0,417.0,31.1],["r",3.0,31.1,417.0,58.9],["r",3.0,58.9,417.0,86.2],["r",3.0,86.2,417.0,112.5],["r",3.0,112.5,417.0,140.3],["r",3.0,140.3,417.0,167.6],["r",3.0,167.6,417.0,193.0],["c",3.0,3.0,177.0,193.0],["c",177.0,3.0,258.1,193.0],["c",258.1,3.0,337.4,193.0],["c",337.4,3.0,417.0,193.0]]},
"1606.09370v1.5":{"paper_id":"1606.09370","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"cff6f8adb3315f0e03b5388957145e0f8020048d314382cd90df11624e217f07","objects":[["t",3.0,3.0,475.0,137.0],["r",3.0,3.0,475.0,31.1],["r",3.0,31.1,475.0,58.9],["r",3.0,58.9,475.0,84.7],["r",3.0,84.7,475.0,110.4],["r",3.0,110.4,475.0,137.0],["c",3.0,3.0,236.4,137.0],["c",236.4,3.0,315.6,137.0],["c",315.6,3.0,394.9,137.0],["c",394.9,3.0,475.0,137.0]]},
"1606.09371v1.2":{"paper_id":"1606.09371","dx":17,"dy":12,"n_rows":7,"n_cols":4,"pixel_sha256":"c1ecab46480e0ddbc9786879c4b972839ef5cb53f0003246a8420583123525b9","objects":[["t",3.0,3.0,475.0,184.0],["r",3.0,3.0,475.0,30.1],["r",3.0,30.1,475.0,55.8],["r",3.0,55.8,475.0,81.6],["r",3.0,81.6,475.0,107.3],["r",3.0,107.3,475.0,133.0],["r",3.0,133.0,475.0,158.8],["r",3.0,158.8,475.0,184.0],["c",3.0,3.0,140.6,184.0],["c",140.6,3.0,261.3,184.0],["c",261.3,3.0,366.2,184.0],["c",366.2,3.0,475.0,184.0]]},
"1606.09371v1.5":{"paper_id":"1606.09371","dx":16,"dy":7,"n_rows":5,"n_cols":4,"pixel_sha256":"c58047166868fbbda58f123c6b47b3844bc66b541d1884804eea6fec62e3cd5a","objects":[["t",3.0,3.0,383.0,133.0],["r",3.0,3.0,383.0,25.1],["r",3.0,25.1,383.0,50.8],["r",3.0,50.8,383.0,76.6],["r",3.0,76.6,383.0,102.3],["r",3.0,102.3,383.0,133.0],["c",3.0,3.0,166.0,133.0],["c",166.0,3.0,238.1,133.0],["c",238.1,3.0,310.3,133.0],["c",310.3,3.0,383.0,133.0]]},
"1609.01344v1.1":{"paper_id":"1609.01344","dx":15,"dy":8,"n_rows":8,"n_cols":2,"pixel_sha256":"f0a82941854339181ef210517b4278cc20654478714cb4d31b6c74b661ef3df2","objects":[["t",3.0,3.0,567.0,142.0],["r",3.0,3.0,567.0,22.4],["r",3.0,22.4,567.0,41.5],["r",3.0,41.5,567.0,58.1],["r",3.0,58.1,567.0,74.7],["r",3.0,74.7,567.0,91.3],["r",3.0,91.3,567.0,107.9],["r",3.0,107.9,567.0,124.5],["r",3.0,124.5,567.0,142.0],["c",3.0,3.0,147.4,142.0],["c",147.4,3.0,567.0,142.0]]},
"1609.01344v1.2":{"paper_id":"1609.01344","dx":16,"dy":11,"n_rows":5,"n_cols":5,"pixel_sha256":"8cb9c077e06a5d2d4ed81c3535a3f14dafd12ee7a463c40488937391729024d2","objects":[["t",3.0,3.0,306.0,134.0],["r",3.0,3.0,306.0,32.7],["r",3.0,32.7,306.0,60.1],["r",3.0,60.1,306.0,85.0],["r",3.0,85.0,306.0,109.9],["r",3.0,109.9,306.0,134.0],["c",3.0,3.0,75.7,134.0],["c",75.7,3.0,132.5,134.0],["c",132.5,3.0,189.9,134.0],["c",189.9,3.0,247.2,134.0],["c",247.2,3.0,306.0,134.0]]},
"1610.06272v2.1":{"paper_id":"1610.06272","dx":16,"dy":10,"n_rows":4,"n_cols":5,"pixel_sha256":"1d45e646e13542ab33e8958ca306df8e4eb9c98e6d4b37735ae8bf2986684d2e","objects":[["t",3.0,3.0,360.0,100.0],["r",3.0,3.0,360.0,28.6],["r",3.0,28.6,360.0,53.9],["r",3.0,53.9,360.0,76.7],["r",3.0,76.7,360.0,100.0],["c",3.0,3.0,60.0,100.0],["c",60.0,3.0,130.7,100.0],["c",130.7,3.0,208.9,100.0],["c",208.9,3.0,279.6,100.0],["c",279.6,3.0,360.0,100.0]]},
"1610.06272v2.2":{"paper_id":"1610.06272","dx":16,"dy":10,"n_rows":4,"n_cols":7,"pixel_sha256":"12af47e92fa608b88fbb3124f0059a04a12346450973d5bbd5ef9d96c85c8fb2","objects":[["t",3.0,3.0,452.0,96.0],["r",3.0,3.0,452.0,26.5],["r",3.0,26.5,452.0,49.7],["r",3.0,49.7,452.0,72.6],["r",3.0,72.6,452.0,96.0],["c",3.0,3.0,59.9,96.0],["c",59.9,3.0,125.3,96.0],["c",125.3,3.0,188.6,96.0],["c",188.6,3.0,251.9,96.0],["c",251.9,3.0,315.2,96.0],["c",315.2,3.0,380.6,96.0],["c",380.6,3.0,452.0,96.0]]},
"1610.06272v2.3":{"paper_id":"1610.06272","dx":16,"dy":10,"n_rows":5,"n_cols":5,"pixel_sha256":"4670255eb46ff1169a9c2ba3be859a0dfc2e97c0182a1dbb08f3bfcce9df59c0","objects":[["t",3.0,3.0,355.0,123.0],["r",3.0,3.0,355.0,26.1],["r",3.0,26.1,355.0,51.4],["r",3.0,51.4,355.0,76.7],["r",3.0,76.7,355.0,99.6],["r",3.0,99.6,355.0,123.0],["c",3.0,3.0,59.9,123.0],["c",59.9,3.0,130.7,123.0],["c",130.7,3.0,201.4,123.0],["c",201.4,3.0,276.5,123.0],["c",276.5,3.0,355.0,123.0],["s",201.4,3.0,355.0,26.1],["s",59.9,3.0,201.4,26.1]]},
"1611.04741v2.1":{"paper_id":"1611.04741","dx":16,"dy":12,"n_rows":8,"n_cols":2,"pixel_sha256":"12d69600e054bb44986fa7e47518b1a87d5a0089452ba0c5a1577521f2b5758f","objects":[["t",3.0,3.0,453.0,214.0],["r",3.0,3.0,453.0,32.1],["r",3.0,32.1,453.0,59.9],["r",3.0,59.9,453.0,85.7],["r",3.0,85.7,453.0,111.4],["r",3.0,111.4,453.0,137.2],["r",3.0,137.2,453.0,162.9],["r",3.0,162.9,453.0,188.6],["r",3.0,188.6,453.0,214.0],["c",3.0,3.0,368.8,214.0],["c",368.8,3.0,453.0,214.0]]},
"1611.04741v2.2":{"paper_id":"1611.04741","dx":16,"dy":11,"n_rows":12,"n_cols":4,"pixel_sha256":"5d946428031580b23a962499cf3031428c4de8d3c5687be4b797d663e96451ca","objects":[["t",3.0,3.0,902.0,321.0],["r",3.0,3.0,902.0,31.1],["r",3.0,31.1,902.0,58.9],["r",3.0,58.9,902.0,84.7],["r",3.0,84.7,902.0,110.4],["r",3.0,110.4,902.0,136.2],["r",3.0,136.2,902.0,161.9],["r",3.0,161.9,902.0,187.6],["r",3.0,187.6,902.0,213.4],["r",3.0,213.4,902.0,239.1],["r",3.0,239.1,902.0,266.9],["r",3.0,266.9,902.0,294.7],["r",3.0,294.7,902.0,321.0],["c",3.0,3.0,376.7,321.0],["c",376.7,3.0,562.8,321.0],["c",562.8,3.0,737.4,321.0],["c",737.4,3.0,902.0,321.0]]},
"1611.04741v2.3":{"paper_id":"1611.04741","dx":16,"dy":11,"n_rows":6,"n_cols":4,"pixel_sha256":"3f9e474e3ebba0a5ee84d98d6ca687fea43fabd7d7ed4d15f7faa15e17a3ceff","objects":[["t",3.0,3.0,580.0,167.0],["r",3.0,3.0,580.0,31.1],["r",3.0,31.1,580.0,58.9],["r",3.0,58.9,580.0,84.7],["r",3.0,84.7,580.0,112.5],["r",3.0,112.5,580.0,140.3],["r",3.0,140.3,580.0,167.0],["c",3.0,3.0,376.7,167.0],["c",376.7,3.0,444.0,167.0],["c",444.0,3.0,511.4,167.0],["c",511.4,3.0,580.0,167.0]]},
"1611.09235v1.1":{"paper_id":"1611.09235","dx":16,"dy":10,"n_rows":7,"n_cols":3,"pixel_sha256":"330d090fef8547f312b608296616ae88e913725a0371968e776b40cb0a571f5b","objects":[["t",3.0,3.0,442.0,165.0],["r",3.0,3.0,442.0,26.5],["r",3.0,26.5,442.0,49.7],["r",3.0,49.7,442.0,72.6],["r",3.0,72.6,442.0,95.4],["r",3.0,95.4,442.0,118.2],["r",3.0,118.2,442.0,141.1],["r",3.0,141.1,442.0,165.0],["c",3.0,3.0,149.3,165.0],["c",149.3,3.0,301.2,165.0],["c",301.2,3.0,442.0,165.0]]},
"1611.09238v1.1":{"paper_id":"1611.09238","dx":16,"dy":11,"n_rows":4,"n_cols":5,"pixel_sha256":"b96b6fdd04fd740c86dd88e32f3d7d58c1c41365be954c83326171d787339519","objects":[["t",3.0,3.0,491.0,99.0],["r",3.0,3.0,491.0,27.5],["r",3.0,27.5,491.0,51.2],["r",3.0,51.2,491.0,74.8],["r",3.0,74.8,491.0,99.0],["c",3.0,3.0,97.0,99.0],["c",97.0,3.0,205.2,99.0],["c",205.2,3.0,293.3,99.0],["c",293.3,3.0,376.6,99.0],["c",376.6,3.0,491.0,99.0]]},
"1702.02925v1.1":{"paper_id":"1702.02925","dx":16,"dy":12,"n_rows":13,"n_cols":3,"pixel_sha256":"11f88fcf82a68acfa1bcd29d39354d92a7d02ad79b8c461a1f848898cbf3d271","objects":[["t",3.0,3.0,613.0,330.0],["r",3.0,3.0,613.0,30.1],["r",3.0,30.1,613.0,55.4],["r",3.0,55.4,613.0,80.3],["r",3.0,80.3,613.0,105.2],["r",3.0,105.2,613.0,130.1],["r",3.0,130.1,613.0,155.0],["r",3.0,155.0,613.0,179.9],["r",3.0,179.9,613.0,204.8],["r",3.0,204.8,613.0,229.7],["r",3.0,229.7,613.0,254.6],["r",3.0,254.6,613.0,279.5],["r",3.0,279.5,613.0,304.4],["r",3.0,304.4,613.0,330.0],["c",3.0,3.0,114.9,330.0],["c",114.9,3.0,335.2,330.0],["c",335.2,3.0,613.0,330.0]]},
"1702.02925v1.2":{"paper_id":"1702.02925","dx":16,"dy":11,"n_rows":4,"n_cols":13,"pixel_sha256":"c3bd82c3ada30ee69508b52269815805c6e6b07e5a9af2486b212b1e58da6aaa","objects":[["t",3.0,3.0,948.0,106.0],["r",3.0,3.0,948.0,29.1],["r",3.0,29.1,948.0,54.4],["r",3.0,54.4,948.0,79.3],["r",3.0,79.3,948.0,106.0],["c",3.0,3.0,205.6,106.0],["c",205.6,3.0,267.4,106.0],["c",267.4,3.0,329.2,106.0],["c",329.2,3.0,391.0,106.0],["c",391.0,3.0,452.8,106.0],["c",452.8,3.0,514.7,106.0],["c",514.7,3.0,576.5,106.0],["c",576.5,3.0,638.3,106.0],["c",638.3,3.0,700.1,106.0],["c",700.1,3.0,761.9,106.0],["c",761.9,3.0,823.7,106.0],["c",823.7,3.0,885.5,106.0],["c",885.5,3.0,948.0,106.0]]},
"1702.02925v1.3":{"paper_id":"1702.02925","dx":15,"dy":11,"n_rows":14,"n_cols":7,"pixel_sha256":"fc407ab9dba9a5c6124ce4bb541a34cb4a751c74bc845cb59e011eba302ba37a","objects":[["t",3.0,3.0,553.0,354.0],["r",3.0,3.0,553.0,29.1],["r",3.0,29.1,553.0,54.4],["r",3.0,54.4,553.0,79.3],["r",3.0,79.3,553.0,104.2],["r",3.0,104.2,553.0,129.1],["r",3.0,129.1,553.0,154.0],["r",3.0,154.0,553.0,178.9],["r",3.0,178.9,553.0,203.8],["r",3.0,203.8,553.0,228.7],["r",3.0,228.7,553.0,253.6],["r",3.0,253.6,553.0,278.5],["r",3.0,278.5,553.0,303.4],["r",3.0,303.4,553.0,328.4],["r",3.0,328.4,553.0,354.0],["c",3.0,3.0,64.4,354.0],["c",64.4,3.0,148.4,354.0],["c",148.4,3.0,230.0,354.0],["c",230.0,3.0,318.1,354.0],["c",318.1,3.0,404.1,354.0],["c",404.1,3.0,482.9,354.0],["c",482.9,3.0,553.0,354.0]]},
"1702.02925v1.6":{"paper_id":"1702.02925","dx":16,"dy":11,"n_rows":10,"n_cols":7,"pixel_sha256":"7b93018a32162cf93026690dff697b9d0a205adec3f7ce2d1160d0146e03efc9","objects":[["t",3.0,3.0,522.0,254.0],["r",3.0,3.0,522.0,29.1],["r",3.0,29.1,522.0,54.4],["r",3.0,54.4,522.0,79.3],["r",3.0,79.3,522.0,104.2],["r",3.0,104.2,522.0,129.1],["r",3.0,129.1,522.0,154.0],["r",3.0,154.0,522.0,178.9],["r",3.0,178.9,522.0,203.8],["r",3.0,203.8,522.0,228.7],["r",3.0,228.7,522.0,254.0],["c",3.0,3.0,65.4,254.0],["c",65.4,3.0,149.4,254.0],["c",149.4,3.0,216.9,254.0],["c",216.9,3.0,305.0,254.0],["c",305.0,3.0,391.0,254.0],["c",391.0,3.0,452.8,254.0],["c",452.8,3.0,522.0,254.0]]},
"1702.02925v1.7":{"paper_id":"1702.02925","dx":16,"dy":11,"n_rows":10,"n_cols":7,"pixel_sha256":"19e70e7985c56361f38f906b2b43a8cdc0c102ac03314d2dff1a49f193d1b570","objects":[["t",3.0,3.0,522.0,254.0],["r",3.0,3.0,522.0,29.1],["r",3.0,29.1,522.0,54.4],["r",3.0,54.4,522.0,79.3],["r",3.0,79.3,522.0,104.2],["r",3.0,104.2,522.0,129.1],["r",3.0,129.1,522.0,154.0],["r",3.0,154.0,522.0,178.9],["r",3.0,178.9,522.0,203.8],["r",3.0,203.8,522.0,228.7],["r",3.0,228.7,522.0,254.0],["c",3.0,3.0,65.4,254.0],["c",65.4,3.0,149.4,254.0],["c",149.4,3.0,216.9,254.0],["c",216.9,3.0,305.0,254.0],["c",305.0,3.0,391.0,254.0],["c",391.0,3.0,452.8,254.0],["c",452.8,3.0,522.0,254.0]]},
"1705.02407v2.1":{"paper_id":"1705.02407","dx":16,"dy":11,"n_rows":8,"n_cols":9,"pixel_sha256":"1e1622f46c133090f5a49069be9879e02a63cce36ffaac035c190aaa8793e176","objects":[["t",3.0,3.0,729.0,194.0],["r",3.0,3.0,729.0,29.6],["r",3.0,29.6,729.0,54.9],["r",3.0,54.9,729.0,77.7],["r",3.0,77.7,729.0,101.0],["r",3.0,101.0,729.0,124.2],["r",3.0,124.2,729.0,147.1],["r",3.0,147.1,729.0,169.9],["r",3.0,169.9,729.0,194.0],["c",3.0,3.0,208.3,194.0],["c",208.3,3.0,277.5,194.0],["c",277.5,3.0,339.8,194.0],["c",339.8,3.0,400.2,194.0],["c",400.2,3.0,462.4,194.0],["c",462.4,3.0,522.6,194.0],["c",522.6,3.0,591.2,194.0],["c",591.2,3.0,657.7,194.0],["c",657.7,3.0,729.0,194.0]]},
"1705.04915v1.2":{"paper_id":"1705.04915","dx":3,"dy":11,"n_rows":4,"n_cols":5,"pixel_sha256":"3421f5de988ea16e5262d3cc0c9f6905f7d4d765acab19781dd70bb504ba5705","objects":[["t",3.0,3.0,421.0,105.0],["r",3.0,3.0,421.0,28.6],["r",3.0,28.6,421.0,53.5],["r",3.0,53.5,421.0,78.5],["r",3.0,78.5,421.0,105.0],["c",3.0,3.0,127.4,105.0],["c",127.4,3.0,216.8,105.0],["c",216.8,3.0,283.4,105.0],["c",283.4,3.0,358.0,105.0],["c",358.0,3.0,421.0,105.0]]},
"1706.08653v2.10":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":4,"n_cols":7,"pixel_sha256":"dd69fc59998a9872fbd23dc10670224cd196aa99d1d401f4548cedf9a1f595a5","objects":[["t",3.0,3.0,508.0,105.0],["r",3.0,3.0,508.0,28.6],["r",3.0,28.6,508.0,55.5],["r",3.0,55.5,508.0,79.3],["r",3.0,79.3,508.0,105.0],["c",3.0,3.0,114.6,105.0],["c",114.6,3.0,180.6,105.0],["c",180.6,3.0,248.4,105.0],["c",248.4,3.0,316.9,105.0],["c",316.9,3.0,379.9,105.0],["c",379.9,3.0,442.3,105.0],["c",442.3,3.0,508.0,105.0],["s",114.6,3.0,180.6,55.5],["s",180.6,3.0,248.4,55.5],["s",316.9,3.0,508.0,28.6],["s",248.4,3.0,316.9,55.5]]},
"1706.08653v2.11":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":8,"n_cols":7,"pixel_sha256":"8a3b16600f91f1bbbb561ab1d5bfe7cd5b05774b5fad49d7b739bef7afe7b324","objects":[["t",3.0,3.0,518.0,207.0],["r",3.0,3.0,518.0,28.6],["r",3.0,28.6,518.0,55.5],["r",3.0,55.5,518.0,79.3],["r",3.0,79.3,518.0,104.6],["r",3.0,104.6,518.0,129.9],["r",3.0,129.9,518.0,155.3],["r",3.0,155.3,518.0,180.6],["r",3.0,180.6,518.0,207.0],["c",3.0,3.0,124.4,207.0],["c",124.4,3.0,190.4,207.0],["c",190.4,3.0,258.2,207.0],["c",258.2,3.0,326.7,207.0],["c",326.7,3.0,389.7,207.0],["c",389.7,3.0,452.1,207.0],["c",452.1,3.0,518.0,207.0],["s",326.7,3.0,518.0,28.6],["s",258.2,3.0,326.7,55.5],["s",190.4,3.0,258.2,55.5],["s",124.4,3.0,190.4,55.5]]},
"1706.08653v2.12":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":8,"n_cols":7,"pixel_sha256":"c78e6546d49fcb9de99cd67fcc7bf3e6412bbe6c8adfa46b93a39f2c5b7a1a79","objects":[["t",3.0,3.0,518.0,207.0],["r",3.0,3.0,518.0,28.6],["r",3.0,28.6,518.0,55.5],["r",3.0,55.5,518.0,79.3],["r",3.0,79.3,518.0,104.6],["r",3.0,104.6,518.0,129.9],["r",3.0,129.9,518.0,155.3],["r",3.0,155.3,518.0,180.6],["r",3.0,180.6,518.0,207.0],["c",3.0,3.0,124.4,207.0],["c",124.4,3.0,190.4,207.0],["c",190.4,3.0,258.2,207.0],["c",258.2,3.0,326.7,207.0],["c",326.7,3.0,389.7,207.0],["c",389.7,3.0,452.1,207.0],["c",452.1,3.0,518.0,207.0],["s",124.4,3.0,190.4,55.5],["s",258.2,3.0,326.7,55.5],["s",190.4,3.0,258.2,55.5],["s",326.7,3.0,518.0,28.6]]},
"1706.08653v2.4":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":7,"n_cols":5,"pixel_sha256":"6a4ebced83a8d94321fcfa5662aefc9dc781271f2c54a43854ceaae61ec757d8","objects":[["t",3.0,3.0,470.0,185.0],["r",3.0,3.0,470.0,31.1],["r",3.0,31.1,470.0,58.5],["r",3.0,58.5,470.0,83.4],["r",3.0,83.4,470.0,108.3],["r",3.0,108.3,470.0,133.2],["r",3.0,133.2,470.0,158.6],["r",3.0,158.6,470.0,185.0],["c",3.0,3.0,152.1,185.0],["c",152.1,3.0,231.4,185.0],["c",231.4,3.0,310.7,185.0],["c",310.7,3.0,389.9,185.0],["c",389.9,3.0,470.0,185.0]]},
"1706.08653v2.5":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":15,"n_cols":5,"pixel_sha256":"283ff8e5bc177a911e1a2c3e003a5b3d0b64fc75b2eaf65ccce083c715ed86ed","objects":[["t",3.0,3.0,512.0,385.0],["r",3.0,3.0,512.0,28.6],["r",3.0,28.6,512.0,54.0],["r",3.0,54.0,512.0,79.3],["r",3.0,79.3,512.0,104.2],["r",3.0,104.2,512.0,129.1],["r",3.0,129.1,512.0,154.0],["r",3.0,154.0,512.0,178.9],["r",3.0,178.9,512.0,204.2],["r",3.0,204.2,512.0,231.4],["r",3.0,231.4,512.0,258.2],["r",3.0,258.2,512.0,283.5],["r",3.0,283.5,512.0,308.8],["r",3.0,308.8,512.0,333.7],["r",3.0,333.7,512.0,359.1],["r",3.0,359.1,512.0,385.0],["c",3.0,3.0,197.4,385.0],["c",197.4,3.0,273.8,385.0],["c",273.8,3.0,353.0,385.0],["c",353.0,3.0,432.3,385.0],["c",432.3,3.0,512.0,385.0],["s",353.0,231.4,512.0,258.2],["s",353.0,3.0,512.0,28.6],["s",197.4,231.4,353.0,258.2],["s",3.0,3.0,197.4,54.0],["s",197.4,3.0,353.0,28.6],["s",3.0,231.4,197.4,283.5]]},
"1706.08653v2.6":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":14,"n_cols":13,"pixel_sha256":"e8a04469e787bc3295883f4a7952273c6a43587091b1fb9dc4efe6344f3b9ef4","objects":[["t",3.0,3.0,933.0,355.0],["r",3.0,3.0,933.0,28.6],["r",3.0,28.6,933.0,55.5],["r",3.0,55.5,933.0,79.3],["r",3.0,79.3,933.0,104.2],["r",3.0,104.2,933.0,129.1],["r",3.0,129.1,933.0,154.0],["r",3.0,154.0,933.0,178.9],["r",3.0,178.9,933.0,203.8],["r",3.0,203.8,933.0,228.7],["r",3.0,228.7,933.0,253.6],["r",3.0,253.6,933.0,278.5],["r",3.0,278.5,933.0,303.9],["r",3.0,303.9,933.0,329.2],["r",3.0,329.2,933.0,355.0],["c",3.0,3.0,124.6,355.0],["c",124.6,3.0,192.0,355.0],["c",192.0,3.0,259.3,355.0],["c",259.3,3.0,326.7,355.0],["c",326.7,3.0,394.0,355.0],["c",394.0,3.0,461.3,355.0],["c",461.3,3.0,528.7,355.0],["c",528.7,3.0,596.0,355.0],["c",596.0,3.0,663.4,355.0],["c",663.4,3.0,730.7,355.0],["c",730.7,3.0,798.0,355.0],["c",798.0,3.0,865.4,355.0],["c",865.4,3.0,933.0,355.0],["s",730.7,3.0,933.0,28.6],["s",124.6,329.2,326.7,355.0],["s",326.7,329.2,528.7,355.0],["s",528.7,329.2,730.7,355.0],["s",730.7,329.2,933.0,355.0],["s",326.7,3.0,528.7,28.6],["s",124.6,3.0,326.7,28.6],["s",528.7,3.0,730.7,28.6]]},
"1706.08653v2.7":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":9,"n_cols":7,"pixel_sha256":"23ae472d525832ba1046b545e4a3d4480a910018606fb18092e4fc07269536ac","objects":[["t",3.0,3.0,538.0,231.0],["r",3.0,3.0,538.0,28.6],["r",3.0,28.6,538.0,55.5],["r",3.0,55.5,538.0,79.3],["r",3.0,79.3,538.0,104.2],["r",3.0,104.2,538.0,129.1],["r",3.0,129.1,538.0,154.0],["r",3.0,154.0,538.0,179.3],["r",3.0,179.3,538.0,205.1],["r",3.0,205.1,538.0,231.0],["c",3.0,3.0,124.6,231.0],["c",124.6,3.0,196.8,231.0],["c",196.8,3.0,264.2,231.0],["c",264.2,3.0,333.6,231.0],["c",333.6,3.0,403.0,231.0],["c",403.0,3.0,470.3,231.0],["c",470.3,3.0,538.0,231.0],["s",124.6,3.0,333.6,28.6],["s",333.6,3.0,538.0,28.6],["s",124.6,205.1,333.6,231.0],["s",333.6,205.1,538.0,231.0],["s",3.0,3.0,124.6,55.5]]},
"1706.08653v2.8":{"paper_id":"1706.08653","dx":16,"dy":11,"n_rows":4,"n_cols":5,"pixel_sha256":"cc1f4cd24da50a1e75669b42b35710b5ddb6a3a411b1fd3be9bab5493c7fba69","objects":[["t",3.0,3.0,442.0,105.0],["r",3.0,3.0,442.0,28.6],["r",3.0,28.6,442.0,54.0],["r",3.0,54.0,442.0,79.3],["r",3.0,79.3,442.0,105.0],["c",3.0,3.0,127.3,105.0],["c",127.3,3.0,203.9,105.0],["c",203.9,3.0,283.2,105.0],["c",283.2,3.0,362.5,105.0],["c",362.5,3.0,442.0,105.0],["s",3.0,3.0,127.3,54.0],["s",127.3,3.0,283.2,28.6],["s",283.2,3.0,442.0,28.6]]},
"1706.10239v2.1":{"paper_id":"1706.10239","dx":15,"dy":11,"n_rows":3,"n_cols":4,"pixel_sha256":"d3a1571d64a8acaaebfc2f980d2956ebd5f68404dc946fc65fb652c0b1de4ff1","objects":[["t",3.0,3.0,584.0,80.0],["r",3.0,3.0,584.0,26.8],["r",3.0,26.8,584.0,52.5],["r",3.0,52.5,584.0,80.0],["c",3.0,3.0,179.1,80.0],["c",179.1,3.0,313.5,80.0],["c",313.5,3.0,448.0,80.0],["c",448.0,3.0,584.0,80.0]]},
"1708.06828v1.5":{"paper_id":"1708.06828","dx":16,"dy":10,"n_rows":7,"n_cols":5,"pixel_sha256":"71bf8f6ca8aca6cc2bfb0b88947ecff1426e3b19b9bd942eee15c03f6df549ff","objects":[["t",3.0,3.0,608.0,183.0],["r",3.0,3.0,608.0,27.6],["r",3.0,27.6,608.0,55.0],["r",3.0,55.0,608.0,82.4],["r",3.0,82.4,608.0,107.3],["r",3.0,107.3,608.0,132.3],["r",3.0,132.3,608.0,157.2],["r",3.0,157.2,608.0,183.0],["c",3.0,3.0,90.6,183.0],["c",90.6,3.0,291.0,183.0],["c",291.0,3.0,461.9,183.0],["c",461.9,3.0,532.9,183.0],["c",532.9,3.0,608.0,183.0],["s",3.0,3.0,90.6,55.0],["s",90.6,3.0,291.0,55.0],["s",291.0,3.0,608.0,27.6]]},
"1709.04959v2.2":{"paper_id":"1709.04959","dx":16,"dy":11,"n_rows":11,"n_cols":4,"pixel_sha256":"18ddbfd4c2d3400a05dbe7b02ae990dd88ce358a8fdefadc84b03e1bb02e9ad9","objects":[["t",3.0,3.0,606.0,257.0],["r",3.0,3.0,606.0,27.5],["r",3.0,27.5,606.0,50.7],["r",3.0,50.7,606.0,73.6],["r",3.0,73.6,606.0,96.4],["r",3.0,96.4,606.0,119.2],["r",3.0,119.2,606.0,142.5],["r",3.0,142.5,606.0,165.7],["r",3.0,165.7,606.0,188.6],["r",3.0,188.6,606.0,211.4],["r",3.0,211.4,606.0,234.2],["r",3.0,234.2,606.0,257.0],["c",3.0,3.0,86.1,257.0],["c",86.1,3.0,328.9,257.0],["c",328.9,3.0,427.2,257.0],["c",427.2,3.0,606.0,257.0]]},
"1709.08718v1.4":{"paper_id":"1709.08718","dx":16,"dy":8,"n_rows":7,"n_cols":2,"pixel_sha256":"824edda886f5b57bb6c87877e277f07c90963cbde52856af33d17ed60c1fe3b0","objects":[["t",3.0,3.0,344.0,170.0],["r",3.0,3.0,344.0,26.1],["r",3.0,26.1,344.0,47.8],["r",3.0,47.8,344.0,76.3],["r",3.0,76.3,344.0,101.2],["r",3.0,101.2,344.0,126.1],["r",3.0,126.1,344.0,151.0],["r",3.0,151.0,344.0,170.0],["c",3.0,3.0,214.6,170.0],["c",214.6,3.0,344.0,170.0]]},
"1709.08718v1.5":{"paper_id":"1709.08718","dx":16,"dy":6,"n_rows":6,"n_cols":4,"pixel_sha256":"0065c3d2ff916ad80e573836f1351f8f8dbfd15917d1596b1e851a17b6e5f7cd","objects":[["t",3.0,2.0,463.0,143.0],["r",3.0,2.0,463.0,24.1],["r",3.0,24.1,463.0,49.4],["r",3.0,49.4,463.0,74.3],["r",3.0,74.3,463.0,99.2],["r",3.0,99.2,463.0,124.1],["r",3.0,124.1,463.0,143.0],["c",3.0,2.0,167.3,143.0],["c",167.3,2.0,246.3,143.0],["c",246.3,2.0,325.0,143.0],["c",325.0,2.0,463.0,143.0]]},
"1711.04434v1.4":{"paper_id":"1711.04434","dx":16,"dy":11,"n_rows":4,"n_cols":4,"pixel_sha256":"cae9d3bd4a5a57c30f5dadb130fde3a2e20130436c6f8fda4cfdb8ee2e03071e","objects":[["t",3.0,3.0,367.0,105.0],["r",3.0,3.0,367.0,29.1],["r",3.0,29.1,367.0,54.4],["r",3.0,54.4,367.0,79.3],["r",3.0,79.3,367.0,105.0],["c",3.0,3.0,159.1,105.0],["c",159.1,3.0,233.1,105.0],["c",233.1,3.0,300.1,105.0],["c",300.1,3.0,367.0,105.0]]},
"1711.04434v1.5":{"paper_id":"1711.04434","dx":16,"dy":11,"n_rows":6,"n_cols":2,"pixel_sha256":"8ec06f5209b98116642aa688630c5b5fb2b25f8957adb72404acbc03e0f8119f","objects":[["t",3.0,3.0,264.0,155.0],["r",3.0,3.0,264.0,25.6],["r",3.0,25.6,264.0,51.1],["r",3.0,51.1,264.0,79.7],["r",3.0,79.7,264.0,104.6],["r",3.0,104.6,264.0,131.0],["r",3.0,131.0,264.0,155.0],["c",3.0,3.0,145.0,155.0],["c",145.0,3.0,264.0,155.0]]},
"1711.04434v1.7":{"paper_id":"1711.04434","dx":16,"dy":11,"n_rows":7,"n_cols":3,"pixel_sha256":"b47cd58dd3955aa6b724fcf7fd36237ef97bc0c5c367c0b01cdf07f98f625f5e","objects":[["t",3.0,3.0,320.0,181.0],["r",3.0,3.0,320.0,29.1],["r",3.0,29.1,320.0,54.4],["r",3.0,54.4,320.0,79.3],["r",3.0,79.3,320.0,104.6],["r",3.0,104.6,320.0,129.9],["r",3.0,129.9,320.0,154.8],["r",3.0,154.8,320.0,181.0],["c",3.0,3.0,106.2,181.0],["c",106.2,3.0,238.9,181.0],["c",238.9,3.0,320.0,181.0],["s",3.0,104.6,106.2,181.0],["s",3.0,29.1,106.2,104.6]]},
"1801.00005v1.1":{"paper_id":"1801.00005","dx":33,"dy":11,"n_rows":6,"n_cols":4,"pixel_sha256":"8aa61bdd7bd4ff8e9527af5ed13d44dede1c8a2883e1377371b9541d2ea43704","objects":[["t",3.0,3.0,496.0,163.0],["r",3.0,3.0,496.0,32.7],["r",3.0,32.7,496.0,58.9],["r",3.0,58.9,496.0,84.7],["r",3.0,84.7,496.0,110.4],["r",3.0,110.4,496.0,136.2],["r",3.0,136.2,496.0,163.0],["c",3.0,3.0,104.1,163.0],["c",104.1,3.0,224.1,163.0],["c",224.1,3.0,356.1,163.0],["c",356.1,3.0,496.0,163.0]]},
"1803.02632v2.1":{"paper_id":"1803.02632","dx":16,"dy":11,"n_rows":4,"n_cols":4,"pixel_sha256":"a1b7ef58d372302c65a3e854679b4a31dd23f9eb949adc2cbd2e34bd3b03568b","objects":[["t",3.0,3.0,811.0,105.0],["r",3.0,3.0,811.0,29.1],["r",3.0,29.1,811.0,54.4],["r",3.0,54.4,811.0,79.3],["r",3.0,79.3,811.0,105.0],["c",3.0,3.0,569.5,105.0],["c",569.5,3.0,651.1,105.0],["c",651.1,3.0,723.2,105.0],["c",723.2,3.0,811.0,105.0]]},
"1803.03670v1.1":{"paper_id":"1803.03670","dx":15,"dy":11,"n_rows":4,"n_cols":5,"pixel_sha256":"f2b8a3c9184f76e246881f7dbe28a5c697fdfe4a778961b62db23690ab61310f","objects":[["t",3.0,3.0,519.0,97.0],["r",3.0,3.0,519.0,27.1],["r",3.0,27.1,519.0,50.3],["r",3.0,50.3,519.0,73.6],["r",3.0,73.6,519.0,97.0],["c",3.0,3.0,76.8,97.0],["c",76.8,3.0,186.5,97.0],["c",186.5,3.0,292.5,97.0],["c",292.5,3.0,407.1,97.0],["c",407.1,3.0,519.0,97.0]]},
"1803.03670v1.3":{"paper_id":"1803.03670","dx":16,"dy":11,"n_rows":6,"n_cols":2,"pixel_sha256":"475383bd4841e7f55226a3cd920a6377f9074d77ccc299c72ae177bf64f0b775","objects":[["t",3.0,3.0,190.0,155.0],["r",3.0,3.0,190.0,28.6],["r",3.0,28.6,190.0,53.5],["r",3.0,53.5,190.0,78.5],["r",3.0,78.5,190.0,103.4],["r",3.0,103.4,190.0,128.7],["r",3.0,128.7,190.0,155.0],["c",3.0,3.0,117.4,155.0],["c",117.4,3.0,190.0,155.0]]},
"1803.07835v1.2":{"paper_id":"1803.07835","dx":16,"dy":12,"n_rows":2,"n_cols":6,"pixel_sha256":"505e4bd3a121dba1c98c7f8f665f790fcba408f933c43ef964032112c78d8853","objects":[["t",3.0,3.0,644.0,56.0],["r",3.0,3.0,644.0,30.1],["r",3.0,30.1,644.0,56.0],["c",3.0,3.0,97.4,56.0],["c",97.4,3.0,174.1,56.0],["c",174.1,3.0,274.6,56.0],["c",274.6,3.0,367.8,56.0],["c",367.8,3.0,511.9,56.0],["c",511.9,3.0,644.0,56.0]]},
"1806.04450v1.1":{"paper_id":"1806.04450","dx":16,"dy":11,"n_rows":7,"n_cols":2,"pixel_sha256":"23b9363e6cb21d1c2ecb0a1f2c59555e3f10b9359711bf3674434834d94fc7c8","objects":[["t",3.0,3.0,332.0,180.0],["r",3.0,3.0,332.0,29.1],["r",3.0,29.1,332.0,54.4],["r",3.0,54.4,332.0,79.3],["r",3.0,79.3,332.0,104.2],["r",3.0,104.2,332.0,129.1],["r",3.0,129.1,332.0,154.0],["r",3.0,154.0,332.0,180.0],["c",3.0,3.0,228.3,180.0],["c",228.3,3.0,332.0,180.0]]},
"1806.04450v1.3":{"paper_id":"1806.04450","dx":4,"dy":9,"n_rows":10,"n_cols":5,"pixel_sha256":"b0c09047bb81ce7d2a587b9f4a4edfe01ad3dc37ee7aadbbb6e27069e2eca185","objects":[["t",3.0,63.0,737.0,317.0],["r",3.0,63.0,737.0,88.9],["r",3.0,88.9,737.0,114.2],["r",3.0,114.2,737.0,139.1],["r",3.0,139.1,737.0,164.0],["r",3.0,164.0,737.0,189.4],["r",3.0,189.4,737.0,215.1],["r",3.0,215.1,737.0,240.4],["r",3.0,240.4,737.0,265.3],["r",3.0,265.3,737.0,290.7],["r",3.0,290.7,737.0,317.0],["c",3.0,63.0,288.6,317.0],["c",288.6,63.0,410.7,317.0],["c",410.7,63.0,531.2,317.0],["c",531.2,63.0,620.4,317.0],["c",620.4,63.0,737.0,317.0]]},
"1807.06535v1.2":{"paper_id":"1807.06535","dx":16,"dy":12,"n_rows":3,"n_cols":4,"pixel_sha256":"0ac7a89e8ff3bb3b5c9a717e1130c0f8916dd24ee681e0dc0c6cb41599542e9d","objects":[["t",3.0,3.0,629.0,105.0],["r",3.0,3.0,629.0,36.3],["r",3.0,36.3,629.0,70.4],["r",3.0,70.4,629.0,105.0],["c",3.0,3.0,87.4,105.0],["c",87.4,3.0,245.3,105.0],["c",245.3,3.0,483.9,105.0],["c",483.9,3.0,629.0,105.0]]},
"1808.00179v1.1":{"paper_id":"1808.00179","dx":15,"dy":11,"n_rows":4,"n_cols":4,"pixel_sha256":"6abb00b361bdec53333dcf014737924df228aad26bb2878719153484f7a32ba8","objects":[["t",3.0,3.0,321.0,107.0],["r",3.0,3.0,321.0,27.2],["r",3.0,27.2,321.0,52.9],["r",3.0,52.9,321.0,78.6],["r",3.0,78.6,321.0,107.0],["c",3.0,3.0,103.6,107.0],["c",103.6,3.0,157.9,107.0],["c",157.9,3.0,238.7,107.0],["c",238.7,3.0,321.0,107.0]]},
"1808.00179v1.11":{"paper_id":"1808.00179","dx":15,"dy":10,"n_rows":3,"n_cols":4,"pixel_sha256":"7e3e9be248d06db1ca3a671784b8d7a974c37b27ff941e7b3ab9f910d58f55ec","objects":[["t",3.0,3.0,321.0,79.0],["r",3.0,3.0,321.0,29.2],["r",3.0,29.2,321.0,53.0],["r",3.0,53.0,321.0,79.0],["c",3.0,3.0,64.1,79.0],["c",64.1,3.0,151.2,79.0],["c",151.2,3.0,254.5,79.0],["c",254.5,3.0,321.0,79.0],["s",3.0,3.0,64.1,53.0],["s",64.1,3.0,321.0,29.2]]},
"1808.00179v1.14":{"paper_id":"1808.00179","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"37487f768ea7a20f4f0be872e69ee8ed5238d8c39bd56496ca016b59228ba523","objects":[["t",3.0,3.0,434.0,132.0],["r",3.0,3.0,434.0,30.2],["r",3.0,30.2,434.0,54.0],["r",3.0,54.0,434.0,79.7],["r",3.0,79.7,434.0,105.4],["r",3.0,105.4,434.0,132.0],["c",3.0,3.0,68.8,132.0],["c",68.8,3.0,190.5,132.0],["c",190.5,3.0,312.2,132.0],["c",312.2,3.0,434.0,132.0],["s",68.8,3.0,434.0,30.2],["s",3.0,3.0,68.8,54.0]]},
"1808.00179v1.3":{"paper_id":"1808.00179","dx":15,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"61e86f0ddaf0796786f136a935d7696c52b105e8e5161c14bc0fb13ab9ed633e","objects":[["t",3.0,3.0,343.0,132.0],["r",3.0,3.0,343.0,30.2],["r",3.0,30.2,343.0,54.0],["r",3.0,54.0,343.0,79.7],["r",3.0,79.7,343.0,105.4],["r",3.0,105.4,343.0,132.0],["c",3.0,3.0,67.8,132.0],["c",67.8,3.0,190.8,132.0],["c",190.8,3.0,266.1,132.0],["c",266.1,3.0,343.0,132.0],["s",3.0,3.0,67.8,54.0],["s",67.8,3.0,343.0,30.2]]},
"1808.00179v1.6":{"paper_id":"1808.00179","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"9462fe171476d288a6524a59ca03f3304bb1f81040cbbf40d82256ce502c8def","objects":[["t",3.0,3.0,434.0,132.0],["r",3.0,3.0,434.0,30.2],["r",3.0,30.2,434.0,54.0],["r",3.0,54.0,434.0,79.7],["r",3.0,79.7,434.0,105.4],["r",3.0,105.4,434.0,132.0],["c",3.0,3.0,68.8,132.0],["c",68.8,3.0,190.5,132.0],["c",190.5,3.0,312.2,132.0],["c",312.2,3.0,434.0,132.0],["s",68.8,3.0,434.0,30.2],["s",3.0,3.0,68.8,54.0]]},
"1808.00179v1.8":{"paper_id":"1808.00179","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"d8e356c260b34373267d0f71f78e5a422ec9b5bed16a04f2d332f5771bc7ad86","objects":[["t",3.0,3.0,482.0,132.0],["r",3.0,3.0,482.0,30.2],["r",3.0,30.2,482.0,54.0],["r",3.0,54.0,482.0,79.7],["r",3.0,79.7,482.0,105.4],["r",3.0,105.4,482.0,132.0],["c",3.0,3.0,68.8,132.0],["c",68.8,3.0,206.4,132.0],["c",206.4,3.0,344.1,132.0],["c",344.1,3.0,482.0,132.0],["s",3.0,3.0,68.8,54.0],["s",68.8,3.0,482.0,30.2]]},
"1808.00179v1.9":{"paper_id":"1808.00179","dx":16,"dy":11,"n_rows":5,"n_cols":4,"pixel_sha256":"5bc5a6bfe5b08bd7fa3d403dc0f233f4bb8e1f939497aa691ce2f5395ab464f4","objects":[["t",3.0,3.0,482.0,132.0],["r",3.0,3.0,482.0,30.2],["r",3.0,30.2,482.0,54.0],["r",3.0,54.0,482.0,79.7],["r",3.0,79.7,482.0,105.4],["r",3.0,105.4,482.0,132.0],["c",3.0,3.0,68.8,132.0],["c",68.8,3.0,206.4,132.0],["c",206.4,3.0,344.1,132.0],["c",344.1,3.0,482.0,132.0],["s",3.0,3.0,68.8,54.0],["s",68.8,3.0,482.0,30.2]]},
"1808.03399v1.1":{"paper_id":"1808.03399","dx":16,"dy":8,"n_rows":7,"n_cols":7,"pixel_sha256":"7e52c59a38a088887301eb00f27cc801e85d0c0b8926a1e2e9c2bc7602ea0c22","objects":[["t",3.0,3.0,596.0,123.0],["r",3.0,3.0,596.0,19.9],["r",3.0,19.9,596.0,36.5],["r",3.0,36.5,596.0,53.6],["r",3.0,53.6,596.0,70.6],["r",3.0,70.6,596.0,87.2],["r",3.0,87.2,596.0,104.2],["r",3.0,104.2,596.0,123.0],["c",3.0,3.0,171.4,123.0],["c",171.4,3.0,247.2,123.0],["c",247.2,3.0,321.5,123.0],["c",321.5,3.0,384.2,123.0],["c",384.2,3.0,460.0,123.0],["c",460.0,3.0,534.3,123.0],["c",534.3,3.0,596.0,123.0],["s",384.2,3.0,596.0,19.9],["s",171.4,3.0,384.2,19.9],["s",384.2,19.9,596.0,36.5],["s",171.4,19.9,384.2,36.5]]}
}"""
if hashlib.sha256(CARRIER_GEOMETRY_JSON.encode()).hexdigest()!=CARRIER_GEOMETRY_SHA256:
    raise RuntimeError("Embedded carrier geometry does not match its SHA-256")
CARRIER_GEOMETRY=json.loads(CARRIER_GEOMETRY_JSON)
CARRIER_LABELS={"t":"table","r":"table row","c":"table column","s":"table spanning cell"}
print({"carrier_tables":len(CARRIER_GEOMETRY),"geometry_sha256":CARRIER_GEOMETRY_SHA256})

In [ ]:
from datasets import load_dataset
from collections import defaultdict
import io

data_files={k:str(v) for k,v in scitsr_paths.items()}
ds=load_dataset("parquet",data_files=data_files)

def norm_text(s):
    # Loose normalization for eligibility checks (headers, row identifiers).
    return re.sub(r"\s+"," ",re.sub(r"[^0-9A-Za-z]+"," ",str(s))).strip().lower()

def image_digest(image):
    rgb=image.convert("RGB")
    return hashlib.sha256(f"{rgb.width}x{rgb.height}:".encode()+rgb.tobytes()).hexdigest()

def place_chunks(chunks,dx,dy,scale=150.0/72.0):
    """Chunk boxes (PDF points, bottom-left origin) in image pixels: the chunks' extent is rendered at
    150 DPI and shifted by the carrier's whole-pixel offset."""
    x_min=min(float(c["x1"]) for c in chunks)
    y_top=max(float(c["y2"]) for c in chunks)
    return [
        [(float(c["x1"])-x_min)*scale+dx,(y_top-float(c["y2"]))*scale+dy,
         (float(c["x2"])-x_min)*scale+dx,(y_top-float(c["y1"]))*scale+dy]
        for c in chunks
    ]

def match_key(s):
    # Alphanumeric characters only; a punctuation-only cell such as "..." keeps its punctuation.
    s=str(s).lower()
    return re.sub(r"[^0-9a-z]+","",s) or re.sub(r"\s+","",s)

def cell_text(cell):
    # Plain cell text; SciTSR's `tex` field holds LaTeX markup that never matches the PDF chunks.
    content=[str(x) for x in (cell.get("content") or []) if str(x).strip()]
    return " ".join(content) if content else str(cell.get("tex") or "")

def match_cells_to_chunks(cells,chunks,chunk_boxes):
    texts=[str(c.get("text","")) for c in chunks]
    # SciTSR's conversion doubles the final character of each table's last chunk ("2.53" -> "2.533").
    if texts and len(texts[-1])>1 and texts[-1][-1]==texts[-1][-2]:
        texts[-1]=texts[-1][:-1]
    keys=[match_key(x) for x in texts]
    used=set()
    out=[]
    for cell in sorted(cells,key=lambda c:(int(c["start_row"]),int(c["start_col"]),int(c.get("id",0)))):
        text=cell_text(cell)
        target=match_key(text)
        if not target:
            continue
        exact=[i for i,k in enumerate(keys) if i not in used and k==target]
        matches=[]
        if exact:
            matches=[exact[0]]
        else:
            # A cell may span several chunks: join unused chunks in reading order until they spell the cell.
            for i in range(len(chunks)):
                if i in used or not keys[i] or not target.startswith(keys[i]): continue
                joined=""
                idxs=[]
                for j in range(i,len(chunks)):
                    if j in used or not keys[j]: continue
                    if not target.startswith(joined+keys[j]): break
                    joined+=keys[j]; idxs.append(j)
                    if joined==target:
                        matches=idxs
                        break
                if matches: break
        if not matches:
            return None
        used.update(matches)
        b=[chunk_boxes[i] for i in matches]
        box=[min(x[0] for x in b),min(x[1] for x in b),max(x[2] for x in b),max(x[3] for x in b)]
        out.append({
            "text":text,
            "row_start":int(cell["start_row"]),"row_end":int(cell["end_row"]),
            "col_start":int(cell["start_col"]),"col_end":int(cell["end_col"]),
            "box":box,
        })
    return out

def decode_source_image(value):
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            image=Image.open(io.BytesIO(value["bytes"]))
            image.load()
            return image.convert("RGB")
        if value.get("path") and Path(value["path"]).is_file():
            image=Image.open(value["path"])
            image.load()
            return image.convert("RGB")
    if isinstance(value,(bytes,bytearray)):
        image=Image.open(io.BytesIO(value))
        image.load()
        return image.convert("RGB")
    raise TypeError(f"Unsupported SciTSR image representation: {type(value).__name__}")

def derive_table(row,geometry):
    table_id=str(row["table_id"])
    image=decode_source_image(row["image"])
    if image_digest(image)!=geometry["pixel_sha256"]:
        raise RuntimeError(f"{table_id}: source pixels differ from the carrier's")
    chunk_boxes=place_chunks(row["chunks"],geometry["dx"],geometry["dy"])
    cell_records=match_cells_to_chunks(row["cells"],row["chunks"],chunk_boxes)
    if not cell_records:
        return None
    n_rows=1+max(c["row_end"] for c in cell_records)
    n_cols=1+max(c["col_end"] for c in cell_records)
    if (n_rows,n_cols)!=(geometry["n_rows"],geometry["n_cols"]):
        return None

    objects=[{"label":CARRIER_LABELS[o[0]],"box":[float(v) for v in o[1:]]} for o in geometry["objects"]]

    grid=[["" for _ in range(n_cols)] for __ in range(n_rows)]
    for c in cell_records:
        if c["row_start"]==c["row_end"] and c["col_start"]==c["col_end"]:
            grid[c["row_start"]][c["col_start"]]=c["text"]

    return {
        "source_id":table_id,
        "paper_id":str(row["paper_id"]),
        "paper_title":str(row["paper_title"]),
        "paper_license":str(row["paper_license"]),
        "image":image,
        "pixel_sha256":geometry["pixel_sha256"],
        "cells":cell_records,
        "objects":objects,
        "n_rows":n_rows,
        "n_cols":n_cols,
        "grid":grid,
        "has_spans":any(c["row_start"]!=c["row_end"] or c["col_start"]!=c["col_end"] for c in cell_records),
        "alignment":{"dx":geometry["dx"],"dy":geometry["dy"]},
    }

derived=[]
failed=[]
for split in ("train","test"):
    for row in ds[split]:
        geometry=CARRIER_GEOMETRY.get(str(row["table_id"]))
        if geometry is None:
            continue  # one of the 14 tables the carrier dropped
        item=derive_table(row,geometry)
        if item is None:
            failed.append(str(row["table_id"]))
        else:
            derived.append(item)

missing=sorted(set(CARRIER_GEOMETRY)-{x["source_id"] for x in derived})
if missing:
    raise RuntimeError(f"{len(missing)} carrier tables failed to derive: {missing[:10]}")

print({
    "source_tables":sum(len(ds[s]) for s in ("train","test")),
    "carrier_tables":len(CARRIER_GEOMETRY),
    "derived_tables":len(derived),
})

## 8. Paper-disjoint split

> **Evaluation practice**

The corpus is split by paper, not by individual table: tables from one paper never appear in two splits.

The notebook reproduces the carrier's split exactly (seed 42, papers drawn in the order test → validation → train):

| split | papers | tables |
|---|---:|---:|
| test | 10 | 21 |
| validation | 7 | 23 |
| train | 29 | 50 |

The notebook performs no training; only the 21 held-out test tables are used below. The next cell stops if the counts differ.

In [ ]:
rng=random.Random(42)
by_paper=defaultdict(list)
for item in derived:
    by_paper[item["paper_id"]].append(item)

papers=sorted(by_paper)
rng.shuffle(papers)
if len(papers)!=46:
    raise RuntimeError(f"Expected the carrier's 46 papers, derived {len(papers)}")

# Iterate the shuffled paper list, never a set: set order depends on per-process string hashing.
test_papers=papers[:10]
val_papers=papers[10:17]
train_papers=papers[17:]

splits={
    "test":[x for p in test_papers for x in by_paper[p]],
    "validation":[x for p in val_papers for x in by_paper[p]],
    "train":[x for p in train_papers for x in by_paper[p]],
}
expected={"test":21,"validation":23,"train":50}
if {k:len(v) for k,v in splits.items()}!=expected:
    raise RuntimeError(f"Paper-disjoint split does not match the carrier: {({k:len(v) for k,v in splits.items()})}")
print({k:{"tables":len(v),"papers":len({x["paper_id"] for x in v})} for k,v in splits.items()})

## 9. Select ten QA-eligible rectangular tables

> **Evaluation practice**

The full end-to-end QA workflow intentionally excludes spanning-cell tables.

Eligibility:

- no spanning cells;
- 3–20 rows;
- 2–12 columns;
- complete first row with unique non-empty headers;
- complete first body column with unique row identifiers;
- at least one numeric body column;
- cells ≤200 characters.

Selection is deterministic and independent of model output.

In [ ]:
NUMBER_RE=re.compile(r"^[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?%?$")

def parse_number(text):
    s=str(text).strip().replace(",","")
    if s.endswith("%"): s=s[:-1]
    if not NUMBER_RE.match(str(text).strip()):
        try:
            # permit comma-stripped representation
            float(s)
        except Exception:
            return None
    try:return float(s)
    except Exception:return None

def eligible(item):
    if item["has_spans"]: return False
    if not (3<=item["n_rows"]<=20 and 2<=item["n_cols"]<=12): return False
    grid=item["grid"]
    if any(not str(x).strip() or len(str(x))>200 for row in grid for x in row): return False
    headers=[norm_text(x) for x in grid[0]]
    if any(not x for x in headers) or len(set(headers))!=len(headers): return False
    row_ids=[norm_text(row[0]) for row in grid[1:]]
    if any(not x for x in row_ids) or len(set(row_ids))!=len(row_ids): return False
    numeric_cols=[]
    for c in range(1,item["n_cols"]):
        vals=[parse_number(row[c]) for row in grid[1:]]
        if len(vals)>=2 and all(v is not None for v in vals):
            numeric_cols.append(c)
    return bool(numeric_cols)

eligible_tables=[x for x in splits["test"] if eligible(x)]
eligible_tables.sort(key=lambda x:hashlib.sha256(f"42:{x['source_id']}".encode()).hexdigest())
canonical_tables=eligible_tables[:END_TO_END_TABLES]
if len(canonical_tables)!=END_TO_END_TABLES:
    # Spec §18: canonical tables come from the paper-disjoint test split only; never borrow train/validation tables.
    raise RuntimeError(f"Only {len(canonical_tables)} QA-eligible test tables were derived")

print([x["source_id"] for x in canonical_tables])

## 10. Create deterministic document pages

> **Evaluation practice**

The real SciTSR table pixels are placed unchanged on a simple white page.

This creates measurable page-level table-detection ground truth without introducing a second table.

In [ ]:
FONT=ImageFont.load_default(size=22)

def make_page(item):
    table=item["image"]
    left,top,right,bottom=120,160,120,120
    page=Image.new("RGB",(table.width+left+right,table.height+top+bottom),"white")
    draw=ImageDraw.Draw(page)
    draw.text((left,48),f"Table {item['source_id']}",font=FONT,fill="black")
    page.paste(table,(left,top))
    draw.text((left,top+table.height+40),"Scientific table excerpt",font=FONT,fill="gray")
    gt_box=[left,top,left+table.width,top+table.height]
    return page,gt_box

for item in canonical_tables:
    item["page"],item["page_gt_box"]=make_page(item)
    item["page_sha256"]=image_digest(item["page"])

print({"pages":len(canonical_tables),"page_sizes":[x["page"].size for x in canonical_tables[:3]]})

## 11. Common detection/structure metrics

> **Evaluation practice**

Bounding-box comparisons use pixel-space IoU.

AP is Pascal-VOC-style all-point AP over score-ranked predictions with one-to-one matching.

In [ ]:
def box_iou(a,b):
    x0=max(float(a[0]),float(b[0])); y0=max(float(a[1]),float(b[1]))
    x1=min(float(a[2]),float(b[2])); y1=min(float(a[3]),float(b[3]))
    inter=max(0,x1-x0)*max(0,y1-y0)
    aa=max(0,float(a[2])-float(a[0]))*max(0,float(a[3])-float(a[1]))
    bb=max(0,float(b[2])-float(b[0]))*max(0,float(b[3])-float(b[1]))
    u=aa+bb-inter
    return inter/u if u>0 else 0.0

def ap_for_label(preds,refs,label,thr):
    det=[]
    n_gt=0
    for i,(p,r) in enumerate(zip(preds,refs,strict=True)):
        gt=[x["box"] for x in r if x["label"]==label]
        n_gt+=len(gt)
        det += [(float(x["score"]),i,x["box"]) for x in p if x["label"]==label]
    det.sort(key=lambda x:-x[0])
    matched=defaultdict(set); hits=[]
    for score,i,box in det:
        gt=[x["box"] for x in refs[i] if x["label"]==label]
        best,bj=0,None
        for j,g in enumerate(gt):
            if j in matched[i]: continue
            v=box_iou(box,g)
            if v>best: best,bj=v,j
        if bj is not None and best>=thr:
            matched[i].add(bj); hits.append(1)
        else:hits.append(0)
    if not hits or n_gt==0:return 0.0
    tp=np.cumsum(hits); recall=tp/n_gt; precision=tp/np.arange(1,len(hits)+1)
    mr=np.concatenate([[0],recall,[1]]); mp=np.concatenate([[0],precision,[0]])
    for i in range(len(mp)-2,-1,-1):mp[i]=max(mp[i],mp[i+1])
    idx=np.where(mr[1:]!=mr[:-1])[0]
    return float(np.sum((mr[idx+1]-mr[idx])*mp[idx+1]))

## 12. Stage 1 — Table Transformer Detection

> **Core concept**

The detector runs on the synthetic page. The highest-scoring surviving `table` / `table rotated` detection becomes the target crop.

A miss is a real pipeline failure and propagates downstream.

**Question tested:** does a detector trained on document pages find a real scientific table placed on a plain synthetic page?

**Predict:** how many of the ten pages will yield a `table` detection at threshold 0.90? Will the best-box IoU be close to 1.0?

In [ ]:
from transformers import AutoImageProcessor, TableTransformerForObjectDetection

t0=time.perf_counter()
det_processor=AutoImageProcessor.from_pretrained(str(DETECTION_DIR),local_files_only=True,trust_remote_code=False)
det_model=TableTransformerForObjectDetection.from_pretrained(
    str(DETECTION_DIR),
    local_files_only=True,
    trust_remote_code=False,
    use_pretrained_backbone=False,
).to(DEVICE).eval()
det_load_seconds=time.perf_counter()-t0
for p in det_model.parameters():p.requires_grad_(False)
det_params=sum(p.numel() for p in det_model.parameters())

def detect_table(image,threshold=DETECTION_THRESHOLD):
    inputs=det_processor(images=image,return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out=det_model(**inputs)
    result=det_processor.post_process_object_detection(
        out,target_sizes=[image.size[::-1]],threshold=float(threshold)
    )[0]
    rows=[]
    for score,label,box in zip(result["scores"],result["labels"],result["boxes"],strict=True):
        name=det_model.config.id2label[int(label)]
        if name in ("table","table rotated"):
            rows.append({"label":name,"score":float(score),"box":[float(v) for v in box.tolist()]})
    rows.sort(key=lambda x:-x["score"])
    return rows

_ = detect_table(canonical_tables[0]["page"])
if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
det_times=[]
for i,item in enumerate(canonical_tables):
    if torch.cuda.is_available():torch.cuda.synchronize()
    t=time.perf_counter(); preds=detect_table(item["page"])
    if torch.cuda.is_available():torch.cuda.synchronize()
    det_times.append(time.perf_counter()-t)
    item["detections"]=preds
    item["selected_detection"]=preds[0] if preds else None
det_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None

print({"mean_latency":float(np.mean(det_times)),"detections":[len(x["detections"]) for x in canonical_tables]})

## 13. Detection metrics

> **Evaluation practice**

Each page has exactly one ground-truth table box.

The tutorial reports AP50/AP75, hit rate and best-box IoU across the ten pages.

In [ ]:
det_pred_lists=[]
det_refs=[]
det_rows=[]
for item in canonical_tables:
    preds=[{"label":"table","score":x["score"],"box":x["box"]} for x in item["detections"]]
    det_pred_lists.append(preds)
    det_refs.append([{"label":"table","box":item["page_gt_box"]}])
    best=max([box_iou(x["box"],item["page_gt_box"]) for x in item["detections"]],default=0.0)
    det_rows.append({
        "table_id":item["source_id"],"predicted":bool(item["detections"]),
        "best_iou":best,"hit50":best>=0.5,"hit75":best>=0.75,
    })

det_metrics={
    "ap50":ap_for_label(det_pred_lists,det_refs,"table",0.5),
    "ap75":ap_for_label(det_pred_lists,det_refs,"table",0.75),
    "hit50":float(np.mean([r["hit50"] for r in det_rows])),
    "hit75":float(np.mean([r["hit75"] for r in det_rows])),
    "mean_best_iou":float(np.mean([r["best_iou"] for r in det_rows])),
    "median_best_iou":float(np.median([r["best_iou"] for r in det_rows])),
}
print(det_metrics)

del det_model,det_processor
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

### What to notice — detection

- `hit50` is the fraction of pages whose best detection overlaps the placed table with IoU ≥ 0.50. A page with no detection above the threshold is a **miss**; every later stage records it as `detection_miss` instead of dropping it, so it still counts against end-to-end accuracy.
- Best-box IoU stays below 1.0 even for a good detection. The reference box is the whole pasted SciTSR image, including its white margin, while the detector tends to hug the ruled table. The red boxes in the §26 panels show the difference.
- AP50 and AP75 are close when every hit is tightly localized; a large gap between them would mean loose boxes.

## 14. Build gold and detected crops

> **Core concept**

The structure recognizer is evaluated twice:

- **gold crop** — exact known table location, with 10 px page padding;
- **detected crop** — predicted box, with the same padding.

Ground-truth structure boxes are transformed into each crop's coordinate frame.

In [ ]:
def padded_crop(page,box,pad=CROP_PADDING):
    x0,y0,x1,y1=[float(v) for v in box]
    cx0=max(0,int(math.floor(x0-pad))); cy0=max(0,int(math.floor(y0-pad)))
    cx1=min(page.width,int(math.ceil(x1+pad))); cy1=min(page.height,int(math.ceil(y1+pad)))
    return page.crop((cx0,cy0,cx1,cy1)),[cx0,cy0,cx1,cy1]

def translate_gold_objects(item,crop_box):
    page_x0,page_y0=120,160
    cx0,cy0,cx1,cy1=crop_box
    out=[]
    for obj in item["objects"]:
        x0,y0,x1,y1=obj["box"]
        b=[x0+page_x0-cx0,y0+page_y0-cy0,x1+page_x0-cx0,y1+page_y0-cy0]
        b=[max(0,b[0]),max(0,b[1]),min(cx1-cx0,b[2]),min(cy1-cy0,b[3])]
        if b[2]>b[0] and b[3]>b[1]:
            out.append({"label":obj["label"],"box":b})
    return out

for item in canonical_tables:
    gold_crop,gold_crop_box=padded_crop(item["page"],item["page_gt_box"])
    item["gold_crop"]=gold_crop; item["gold_crop_box"]=gold_crop_box
    item["gold_crop_objects"]=translate_gold_objects(item,gold_crop_box)

    if item["selected_detection"] is not None:
        det_crop,det_crop_box=padded_crop(item["page"],item["selected_detection"]["box"])
        item["det_crop"]=det_crop; item["det_crop_box"]=det_crop_box
        item["det_crop_objects"]=translate_gold_objects(item,det_crop_box)
    else:
        item["det_crop"]=None; item["det_crop_box"]=None; item["det_crop_objects"]=[]

## 15. Stage 2 — Table Transformer Structure Recognition

> **Core concept**

Raw model output is preserved.

Rows and columns are later subjected to deterministic same-label NMS for grid reconstruction. NMS is a composition rule of this notebook, not part of the model.

**Question tested:** does structure recognition depend on the crop it is given?

**Predict:** will the gold-crop path or the detected-crop path produce more exact grids (row count and column count both right)?

In [ ]:
struct_processor=AutoImageProcessor.from_pretrained(
    str(STRUCTURE_DIR),
    local_files_only=True,
    trust_remote_code=False,
    size={"shortest_edge":800,"longest_edge":800},
)
t0=time.perf_counter()
struct_model=TableTransformerForObjectDetection.from_pretrained(
    str(STRUCTURE_DIR),
    local_files_only=True,
    trust_remote_code=False,
    use_pretrained_backbone=False,
).to(DEVICE).eval()
struct_load_seconds=time.perf_counter()-t0
for p in struct_model.parameters():p.requires_grad_(False)
struct_params=sum(p.numel() for p in struct_model.parameters())

def recognize_structure(image,threshold=STRUCTURE_THRESHOLD):
    inputs=struct_processor(images=image,return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out=struct_model(**inputs)
    result=struct_processor.post_process_object_detection(
        out,target_sizes=[image.size[::-1]],threshold=float(threshold)
    )[0]
    rows=[]
    for score,label,box in zip(result["scores"],result["labels"],result["boxes"],strict=True):
        name=struct_model.config.id2label[int(label)]
        rows.append({"label":name,"score":float(score),"box":[float(v) for v in box.tolist()]})
    rows.sort(key=lambda x:-x["score"])
    return rows

_ = recognize_structure(canonical_tables[0]["gold_crop"])
if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
struct_times=[]
for item in canonical_tables:
    t=time.perf_counter()
    item["gold_structure_raw"]=recognize_structure(item["gold_crop"])
    if item["det_crop"] is not None:
        item["det_structure_raw"]=recognize_structure(item["det_crop"])
    else:
        item["det_structure_raw"]=[]
    struct_times.append(time.perf_counter()-t)
struct_peak=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
print({"mean_pair_seconds":float(np.mean(struct_times))})

## 16. Grid reconstruction

> **Core concept**

Rows are sorted top-to-bottom; columns left-to-right.

Same-label duplicate boxes are removed with NMS at IoU 0.50 before grid construction.

In [ ]:
def nms_items(items,iou_threshold=GRID_NMS_IOU):
    kept=[]
    for item in sorted(items,key=lambda x:-x["score"]):
        if all(box_iou(item["box"],k["box"])<iou_threshold for k in kept):
            kept.append(item)
    return kept

def grid_from_structure(raw):
    rows=nms_items([x for x in raw if x["label"]=="table row"])
    cols=nms_items([x for x in raw if x["label"]=="table column"])
    rows.sort(key=lambda x:(x["box"][1]+x["box"][3])/2)
    cols.sort(key=lambda x:(x["box"][0]+x["box"][2])/2)
    cells=[]
    valid=bool(rows and cols)
    for ri,r in enumerate(rows):
        row=[]
        for ci,c in enumerate(cols):
            b=[
                max(r["box"][0],c["box"][0]),max(r["box"][1],c["box"][1]),
                min(r["box"][2],c["box"][2]),min(r["box"][3],c["box"][3]),
            ]
            if b[2]<=b[0] or b[3]<=b[1]:valid=False
            row.append(b)
        cells.append(row)
    return {"rows":rows,"columns":cols,"cells":cells,"valid_geometry":valid}

for item in canonical_tables:
    item["gold_grid_pred"]=grid_from_structure(item["gold_structure_raw"])
    item["det_grid_pred"]=grid_from_structure(item["det_structure_raw"]) if item["det_crop"] is not None else {"rows":[],"columns":[],"cells":[],"valid_geometry":False}

## 17. Assign SciTSR source cell text to predicted cells

> **Core concept**

For each gold logical cell, its pixel center is transformed into the current crop and located in the predicted row and column.

No gold row/column index is supplied to the reconstruction algorithm.

In [ ]:
def source_cells_in_crop(item,crop_box):
    cx0,cy0,_,_=crop_box
    page_left,page_top=120,160
    out=[]
    for c in item["cells"]:
        b=c["box"]
        out.append({
            **c,
            "crop_box":[b[0]+page_left-cx0,b[1]+page_top-cy0,b[2]+page_left-cx0,b[3]+page_top-cy0],
        })
    return out

def contains(box,x,y):
    return box[0]<=x<=box[2] and box[1]<=y<=box[3]

def reconstruct_table(item,grid,crop_box):
    if not grid["valid_geometry"] or not grid["rows"] or not grid["columns"]:
        return {"valid":False,"reason":"grid_invalid","table":None,"mapped":0,"unmapped":len(item["cells"]),"ambiguous":0}
    strings=[[[] for _ in grid["columns"]] for __ in grid["rows"]]
    mapped=unmapped=ambiguous=correct=0
    simple=[c for c in source_cells_in_crop(item,crop_box) if c["row_start"]==c["row_end"] and c["col_start"]==c["col_end"]]
    for c in simple:
        b=c["crop_box"]; x=(b[0]+b[2])/2; y=(b[1]+b[3])/2
        ris=[i for i,r in enumerate(grid["rows"]) if contains(r["box"],x,y)]
        cis=[i for i,col in enumerate(grid["columns"]) if contains(col["box"],x,y)]
        if len(ris)==1 and len(cis)==1:
            ri,ci=ris[0],cis[0]
            strings[ri][ci].append(c["text"])
            mapped+=1
            if ri==c["row_start"] and ci==c["col_start"]:correct+=1
        elif len(ris)==0 or len(cis)==0:
            unmapped+=1
        else:
            ambiguous+=1
    table=[[" ".join(cell).strip() for cell in row] for row in strings]
    headers=table[0] if table else []
    norm_headers=[norm_text(x) for x in headers]
    valid_headers=bool(headers) and all(norm_headers) and len(set(norm_headers))==len(norm_headers)
    valid=grid["valid_geometry"] and valid_headers and len(table)<=64 and len(headers)<=32
    return {
        "valid":valid,
        "reason":"ok" if valid else "header_or_limit_invalid",
        "table":table,
        "mapped":mapped,"unmapped":unmapped,"ambiguous":ambiguous,
        "assignment_accuracy":correct/max(1,len(simple)),
        "headers_valid":valid_headers,
    }

for item in canonical_tables:
    item["gold_recon"]=reconstruct_table(item,item["gold_grid_pred"],item["gold_crop_box"])
    if item["det_crop"] is not None:
        item["det_recon"]=reconstruct_table(item,item["det_grid_pred"],item["det_crop_box"])
    else:
        item["det_recon"]={"valid":False,"reason":"detection_miss","table":None,"mapped":0,"unmapped":len(item["cells"]),"ambiguous":0}

## 18. Structure and reconstruction metrics

> **Evaluation practice**

Rows/columns are scored independently on gold crops and detected crops.

Header predictions are not penalized because SciTSR-PD does not annotate the two PubTables header classes used by the structure checkpoint.

In [ ]:
def refs_for(item,key,label):
    return [x for x in item[key] if x["label"]==label]

def preds_for(item,key,label):
    return [x for x in item[key] if x["label"]==label]

structure_rows=[]
for path_name,pred_key,ref_key in [
    ("gold_crop","gold_structure_raw","gold_crop_objects"),
    ("detected_crop","det_structure_raw","det_crop_objects"),
]:
    for item in canonical_tables:
        gp=item["gold_grid_pred"] if path_name=="gold_crop" else item["det_grid_pred"]
        gold_rows=refs_for(item,ref_key,"table row")
        gold_cols=refs_for(item,ref_key,"table column")
        pred_rows=preds_for(item,pred_key,"table row")
        pred_cols=preds_for(item,pred_key,"table column")
        best_r=[max([box_iou(g["box"],p["box"]) for p in pred_rows],default=0) for g in gold_rows]
        best_c=[max([box_iou(g["box"],p["box"]) for p in pred_cols],default=0) for g in gold_cols]
        structure_rows.append({
            "table_id":item["source_id"],"path":path_name,
            "gold_rows":len(gold_rows),"predicted_rows":len(gp["rows"]),
            "gold_columns":len(gold_cols),"predicted_columns":len(gp["columns"]),
            "row_count_exact":len(gp["rows"])==len(gold_rows),
            "column_count_exact":len(gp["columns"])==len(gold_cols),
            "grid_shape_exact":len(gp["rows"])==len(gold_rows) and len(gp["columns"])==len(gold_cols),
            "mean_row_iou":float(np.mean(best_r)) if best_r else 0.0,
            "mean_column_iou":float(np.mean(best_c)) if best_c else 0.0,
        })

for path_name in ("gold_crop","detected_crop"):
    subset=[r for r in structure_rows if r["path"]==path_name]
    print(path_name,{
        "grid_exact":float(np.mean([r["grid_shape_exact"] for r in subset])),
        "mean_row_iou":float(np.mean([r["mean_row_iou"] for r in subset])),
        "mean_column_iou":float(np.mean([r["mean_column_iou"] for r in subset])),
    })

### What to notice — structure

- `grid_exact` is the gate for reconstruction: when the row or column count is wrong, text is assigned to the wrong cells no matter how good the boxes look.
- Mean row/column IoU compares each reference row or column with its best-matching prediction. Exact counts with IoU well below 1.0 are normal here: the carrier's reference rows and columns tile the whole table from mid-gap to mid-gap (the PubTables-1M convention), while predicted boxes follow the model's own boundaries. Boundary offsets matter only when a cell's center falls on the wrong side of a boundary.
- The detected-crop path can lose a table entirely (a detection miss gives an empty structure) or gain or lose a row or column when the crop clips or adds margin.
- Column-header and projected-row-header predictions are not scored, because SciTSR-PD does not annotate them.

### Checkpoint — structure

1. A predicted grid has the right number of columns but one row too few. What happens to the text of the cells in the missing row?
2. Why can a detected crop change the structure result even when its detection IoU is above 0.8?

<details>
<summary>Sample answer</summary>

1. The centers of the missing row's cells fall into a neighbouring predicted row, and their text is merged into that row's cells, or into no row at all, and they become unmapped. Either way the reconstructed table no longer says what the source table says, so questions about that row can fail although TAPAS itself is unchanged.
2. The structure model sees a different image. The crop's margins, and therefore its scale after resizing to 800 px, change, and a clipped rule or header line removes evidence. IoU measures overlap with the reference box, not whether every rule and header line is inside the crop.

</details>

### Activity — Predict → Change one thing → Run → Observe → Explain

This activity uses the 23 **validation** tables only, so the canonical test-table results above and the exported files are unchanged.

1. **Predict.** If the structure threshold rises from 0.50 to `ACTIVITY_STRUCTURE_THRESHOLD`, will the model report more rows and columns, fewer, or the same? Which kinds of tables will change?
2. **Change one thing.** Set `ACTIVITY_STRUCTURE_THRESHOLD` in the next cell (default 0.95; then try 0.98 and 0.30).
3. **Run** the cell. It recognizes structure on each validation table's gold crop at both thresholds.
4. **Observe** the exact-grid counts and the tables whose row/column counts changed.
5. **Explain** the tradeoff: what does a higher threshold remove, and what does that cost?

<details>
<summary>Sample answer</summary>

A higher threshold keeps only the more confident boxes, so it can only remove predictions. On these clean, ruled scientific tables the model is confident about nearly every real row and column, so lowering the threshold changes little, while raising it toward 1.0 starts removing real rows or columns — typically short or sparse ones — and the grid comes out a row or column short. On noisier documents a lower threshold can also admit spurious or duplicated boxes that NMS must then suppress. Because reconstruction needs exact counts, the useful threshold is the one that gets counts right most often on held-out validation tables. That is why the experiment runs on validation tables, not on the test tables the notebook reports.

</details>

In [ ]:
# Change one thing: the structure threshold, on validation tables only (canonical outputs are unaffected).
ACTIVITY_STRUCTURE_THRESHOLD = 0.95  # @param {type:"number"}
if not 0 <= ACTIVITY_STRUCTURE_THRESHOLD <= 1:
    raise ValueError("ACTIVITY_STRUCTURE_THRESHOLD must be in [0,1]")

def grid_counts(crop,threshold):
    grid=grid_from_structure(recognize_structure(crop,threshold=threshold))
    return len(grid["rows"]),len(grid["columns"])

activity_rows=[]
for item in splits["validation"]:
    page,page_box=make_page(item)
    crop,_=padded_crop(page,page_box)
    activity_rows.append({
        "table":item["source_id"],
        "gold":(item["n_rows"],item["n_cols"]),
        "at_reference":grid_counts(crop,STRUCTURE_THRESHOLD),
        "at_changed":grid_counts(crop,ACTIVITY_STRUCTURE_THRESHOLD),
    })

for name,threshold in (("at_reference",STRUCTURE_THRESHOLD),("at_changed",ACTIVITY_STRUCTURE_THRESHOLD)):
    exact=sum(r[name]==r["gold"] for r in activity_rows)
    print(f"threshold {threshold:.2f}: exact grids {exact} / {len(activity_rows)} validation tables")
changed=[r for r in activity_rows if r["at_reference"]!=r["at_changed"]]
print(f"Tables whose (rows, columns) changed: {len(changed)}")
for r in changed:
    print(" ",r)

## 19. Complex spanning-cell probe

> **Core concept**

The first three held-out spanning-cell tables are run through structure recognition on their gold crops. Each panel in `complex_probe/` shows the gold spanning cells beside the predicted rows, columns and spanning cells.

They are intentionally excluded from rectangular TAPAS reconstruction because simple row × column intersections cannot faithfully represent their semantics.

In [ ]:
complex_tables=[x for x in splits["test"] if x["has_spans"]][:COMPLEX_STRUCTURE_TABLES] if RUN_COMPLEX_STRUCTURE_PROBE else []
complex_dir=Path(OUTPUT_DIR)/"complex_probe"
complex_dir.mkdir(parents=True,exist_ok=True)
PROBE_COLORS={"table row":"royalblue","table column":"seagreen","table spanning cell":"magenta"}

def outline(image,items,colors):
    out=image.convert("RGB").copy()
    d=ImageDraw.Draw(out)
    for x in items:
        d.rectangle(x["box"],outline=colors.get(x["label"],"orange"),width=2)
    return out

for item in complex_tables:
    page,page_box=make_page(item)
    crop,crop_box=padded_crop(page,page_box)
    gold_spans=[x for x in translate_gold_objects(item,crop_box) if x["label"]=="table spanning cell"]
    raw=recognize_structure(crop)
    grid=grid_from_structure(raw)
    pred_spans=[x for x in raw if x["label"]=="table spanning cell"]
    left=outline(crop,gold_spans,{"table spanning cell":"orange"})
    right=outline(crop,grid["rows"]+grid["columns"]+pred_spans,PROBE_COLORS)
    panel=Image.new("RGB",(left.width+right.width+20,left.height+60),"white")
    panel.paste(left,(0,40)); panel.paste(right,(left.width+20,40))
    d=ImageDraw.Draw(panel)
    d.text((5,8),"gold spanning cells (orange)",fill="black")
    d.text((left.width+25,8),"predicted rows / columns / spanning cells",fill="black")
    panel.save(complex_dir/f"{item['source_id'].replace('/','_')}.png")
    print({
        "table":item["source_id"],
        "gold_grid":[item["n_rows"],item["n_cols"]],"predicted_grid":[len(grid["rows"]),len(grid["columns"])],
        "gold_spanning_cells":len(gold_spans),"predicted_spanning_cells":len(pred_spans),
    })
if complex_tables:
    print("These tables are intentionally excluded from the simple rectangular QA reconstruction contract.")

# The structure recognizer is no longer needed: unload it before TAPAS (sequential loading, spec §67).
del struct_model,struct_processor
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

## 20. Generate 50 table-QA questions from the gold logical tables

> **Evaluation practice**

Each table contributes:

- 2 lookup (`NONE`) questions;
- 1 `SUM`;
- 1 `AVERAGE`;
- 1 `COUNT`.

Generation uses only the gold logical table and happens independently of model predictions.

In [ ]:
def gold_table_dict(item):
    headers=item["grid"][0]
    body=item["grid"][1:]
    return {headers[c]:[row[c] for row in body] for c in range(len(headers))}

def numeric_columns(item):
    out=[]
    for c in range(1,item["n_cols"]):
        vals=[parse_number(row[c]) for row in item["grid"][1:]]
        if len(vals)>=2 and all(v is not None for v in vals):
            out.append((c,vals))
    return out

qa_records=[]
for item in canonical_tables:
    grid=item["grid"]; headers=grid[0]; body=grid[1:]
    row_ids=[row[0] for row in body]
    candidates=[]
    for r in range(len(body)):
        for c in range(1,len(headers)):
            candidates.append((r,c))
    candidates.sort(key=lambda rc:hashlib.sha256(f"{item['source_id']}:{rc[0]}:{rc[1]}".encode()).hexdigest())
    for j,(r,c) in enumerate(candidates[:2]):
        qa_records.append({
            "id":f"{item['source_id']}-lookup-{j}",
            "table_id":item["source_id"],
            "question":f"What is the {headers[c]} for {row_ids[r]}?",
            "category":"NONE",
            "gold_aggregation":"NONE",
            "gold_coordinates":[[r,c]],
            "gold_denotation":[body[r][c]],
        })
    c,vals=numeric_columns(item)[0]
    coords=[[r,c] for r in range(len(body))]
    qa_records += [
        {
            "id":f"{item['source_id']}-sum","table_id":item["source_id"],
            "question":f"What is the total {headers[c]}?","category":"SUM",
            "gold_aggregation":"SUM","gold_coordinates":coords,
            "gold_denotation":float(sum(vals)),
        },
        {
            "id":f"{item['source_id']}-avg","table_id":item["source_id"],
            "question":f"What is the average {headers[c]}?","category":"AVERAGE",
            "gold_aggregation":"AVERAGE","gold_coordinates":coords,
            "gold_denotation":float(sum(vals)/len(vals)),
        },
        {
            "id":f"{item['source_id']}-count","table_id":item["source_id"],
            "question":f"How many {headers[c]} entries are listed?","category":"COUNT",
            "gold_aggregation":"COUNT","gold_coordinates":coords,
            "gold_denotation":float(len(vals)),
        },
    ]

if len(qa_records)!=50:
    raise RuntimeError(f"Expected 50 questions, got {len(qa_records)}")
counts=defaultdict(int)
for r in qa_records:counts[r["category"]]+=1
if dict(counts)!={"NONE":20,"SUM":10,"AVERAGE":10,"COUNT":10}:
    raise RuntimeError(f"Question mix mismatch: {dict(counts)}")
print(dict(counts))

## 21. Stage 3 — TAPAS

> **Core concept**

TAPAS receives a string table and a question. It selects cells at threshold 0.5 and predicts one aggregation operator.

For `SUM`, `AVERAGE` and `COUNT`, the notebook computes the numeric answer from selected cells exactly as the DIMER reference carrier does.

**Question tested:** how much QA accuracy is lost when TAPAS reads a reconstructed table instead of the gold one?

**Predict:** rank the three paths (gold table, structure-only, end-to-end) by denotation accuracy, and estimate the gap between the first and the last.

In [ ]:
from transformers import TapasTokenizer, TapasForQuestionAnswering

tapas_tokenizer=TapasTokenizer.from_pretrained(str(TAPAS_DIR),local_files_only=True)
t0=time.perf_counter()
tapas_model=TapasForQuestionAnswering.from_pretrained(
    str(TAPAS_DIR),local_files_only=True,dtype=torch.float32
).to(DEVICE).eval()
tapas_load_seconds=time.perf_counter()-t0
for p in tapas_model.parameters():p.requires_grad_(False)
tapas_params=sum(p.numel() for p in tapas_model.parameters())
AGGREGATIONS=("NONE","SUM","AVERAGE","COUNT")

def compute_numeric(cells,aggregation):
    vals=[parse_number(x) for x in cells]
    if aggregation=="COUNT":return float(len(cells))
    if any(v is None for v in vals):return None
    if aggregation=="SUM":return float(sum(vals))
    if aggregation=="AVERAGE":return float(sum(vals)/len(vals)) if vals else None
    return None

def table_to_df(table):
    headers=table[0]
    body=table[1:]
    return pd.DataFrame(body,columns=headers,dtype=object)

def tapas_answer(table,question):
    df=table_to_df(table)
    inputs=tapas_tokenizer(
        table=df,
        queries=[question],
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    model_inputs={k:v.to(DEVICE) for k,v in inputs.items()}
    with torch.inference_mode():
        out=tapas_model(**model_inputs)
    coords,agg=tapas_tokenizer.convert_logits_to_predictions(
        inputs,
        out.logits.detach().cpu(),
        out.logits_aggregation.detach().cpu(),
    )
    selected=[list(x) for x in coords[0]]
    aggregation=AGGREGATIONS[int(agg[0])]
    cells=[str(df.iat[r,c]) for r,c in selected]
    numeric=compute_numeric(cells,aggregation)
    answer=(f"{aggregation} > " if aggregation!="NONE" else "")+", ".join(cells)
    return {
        "coordinates":selected,"cells":cells,"aggregation":aggregation,
        "numeric_answer":numeric,"answer":answer,
    }

print({"parameters":tapas_params,"load_seconds":tapas_load_seconds})

## 22. QA scoring

> **Evaluation practice**

Denotation correctness is the headline downstream metric:

- lookup: same multiset of normalized selected cell strings;
- numeric aggregation: numeric answer matches within `1e-6`.

Gold-table evaluation also reports aggregation and exact selected-coordinate accuracy.

In [ ]:
def norm_cell(s):
    return " ".join(str(s).lower().split())

def denotation_correct(result,record):
    gold=record["gold_denotation"]
    if isinstance(gold,list):
        return sorted(norm_cell(x) for x in result["cells"])==sorted(norm_cell(x) for x in gold)
    value=result["numeric_answer"]
    return value is not None and abs(float(value)-float(gold))<=1e-6

def score_path(path_name,table_provider):
    rows=[]
    times=[]
    for record in qa_records:
        item=next(x for x in canonical_tables if x["source_id"]==record["table_id"])
        table,status=table_provider(item)
        if table is None:
            rows.append({
                "question_id":record["id"],"table_id":record["table_id"],"path":path_name,
                "question":record["question"],"gold_aggregation":record["gold_aggregation"],
                "predicted_aggregation":None,"gold_denotation":record["gold_denotation"],
                "predicted_denotation":None,"denotation_correct":False,
                "aggregation_correct":False,"selected_cells":[],"coordinates":[],
                "pipeline_status":status,"latency_seconds":None,
            })
            continue
        t=time.perf_counter()
        result=tapas_answer(table,record["question"])
        times.append(time.perf_counter()-t)
        rows.append({
            "question_id":record["id"],"table_id":record["table_id"],"path":path_name,
            "question":record["question"],"gold_aggregation":record["gold_aggregation"],
            "predicted_aggregation":result["aggregation"],"gold_denotation":record["gold_denotation"],
            "predicted_denotation":result["numeric_answer"] if record["gold_aggregation"]!="NONE" else result["cells"],
            "denotation_correct":denotation_correct(result,record),
            "aggregation_correct":result["aggregation"]==record["gold_aggregation"],
            "selected_cells":result["cells"],"coordinates":result["coordinates"],
            "pipeline_status":"ok","latency_seconds":times[-1],
        })
    return rows,times

def gold_provider(item):
    return item["grid"],"ok"

def struct_provider(item):
    r=item["gold_recon"]
    return (r["table"],"ok") if r["valid"] else (None,r["reason"])

def full_provider(item):
    if item["selected_detection"] is None:return None,"detection_miss"
    r=item["det_recon"]
    return (r["table"],"ok") if r["valid"] else (None,r["reason"])

gold_qa,gold_tapas_times=score_path("gold_table",gold_provider)
structure_qa,structure_tapas_times=score_path("structure_only",struct_provider)
end_to_end_qa,end_tapas_times=score_path("end_to_end",full_provider)

def qa_summary(rows):
    return {
        "n":len(rows),
        "denotation_accuracy":float(np.mean([r["denotation_correct"] for r in rows])),
        "aggregation_accuracy":float(np.mean([r["aggregation_correct"] for r in rows])),
        "valid_questions":sum(r["pipeline_status"]=="ok" for r in rows),
    }

print("Gold:",qa_summary(gold_qa))
print("Structure-only:",qa_summary(structure_qa))
print("End-to-end:",qa_summary(end_to_end_qa))

### What to notice — QA

- The gold-table path is TAPAS's own ceiling on these questions. Its errors are QA-model errors (wrong cells or wrong aggregation), not pipeline errors.
- The structure-only path matches the gold path only when every gold-crop reconstruction is valid and puts each cell's text in the right place; any drop below the gold path was introduced by structure recognition or reconstruction.
- The end-to-end path additionally pays for detection misses and detected-crop structure errors. Questions whose table could not be reconstructed count as wrong (`pipeline_status` records why), so all three paths share the same 50-question denominator.
- A lookup answer counts as correct when the selected cells are right, even if TAPAS also attached an aggregation operator, so aggregation accuracy can sit below denotation accuracy.

## 23. Pipeline waterfall

> **Evaluation practice**

The waterfall counts failures rather than dropping them from downstream denominators.

In [ ]:
waterfall={
    "tables":len(canonical_tables),
    "detection":{
        "hit50":sum(r["hit50"] for r in det_rows),
        "hit75":sum(r["hit75"] for r in det_rows),
    },
    "structure_gold_crop":{
        "grid_exact":sum(r["grid_shape_exact"] for r in structure_rows if r["path"]=="gold_crop"),
    },
    "structure_detected_crop":{
        "grid_exact":sum(r["grid_shape_exact"] for r in structure_rows if r["path"]=="detected_crop"),
    },
    "reconstruction":{
        "valid_gold_crop":sum(x["gold_recon"]["valid"] for x in canonical_tables),
        "valid_detected_crop":sum(x["det_recon"]["valid"] for x in canonical_tables),
    },
    "qa":{
        "questions":50,
        "gold_table_denotation_accuracy":qa_summary(gold_qa)["denotation_accuracy"],
        "structure_only_denotation_accuracy":qa_summary(structure_qa)["denotation_accuracy"],
        "end_to_end_denotation_accuracy":qa_summary(end_to_end_qa)["denotation_accuracy"],
    },
}
print(json.dumps(waterfall,indent=2))

### What to notice — waterfall

Read the waterfall top to bottom. Along the end-to-end path, the counts generally shrink from detection hits, to exact detected-crop grids, to valid detected-crop tables, to correct answers. The step with the largest drop is where most of the end-to-end loss entered. Compare each gold-crop count with its detected-crop count to separate structure errors from errors caused by the crop.

## 24. Checkpoint — trace the losses

> **Evaluation practice**

Return to the predictions you wrote down in §2. The next cell prints, for each canonical table, the detection IoU, whether each path produced a valid table, the predicted grid shapes, and the cell-assignment accuracy on each crop. Compare them with the waterfall above.

1. Which stage reduced end-to-end accuracy the most in this run?
2. Can perfect table detection guarantee correct structure?
3. Can perfect structure guarantee correct QA?
4. Why does TAPAS need strings rather than row/column boxes?
5. Where would OCR enter a production version of this workflow?
6. What happens if one predicted row is missing?
7. What happens if the first reconstructed row is mistaken?

<details>
<summary>Sample answer</summary>

1. Look for the largest step in the waterfall. When the gold-crop path is nearly perfect, the loss enters with detection (misses) and with detected-crop structure errors (a row or column gained or lost through the crop).
2. No. Structure recognition sees only the crop; margins, clipped rules and faint rows can still change the predicted grid.
3. No. The gold-table path is the ceiling: TAPAS still selects wrong cells or the wrong aggregation on some questions with a perfect table.
4. TAPAS is a language model over a table of strings: it embeds cell text together with row/column position ids. Boxes carry no words, so something must place text into the grid first.
5. Between structure recognition and reconstruction: OCR or PDF text extraction supplies the words and their boxes, which are then assigned to predicted cells. This notebook uses SciTSR annotation text in that position.
6. The texts of that row merge into a neighbouring row or become unmapped; the table changes meaning, and questions about the row fail.
7. The first row becomes the header. With a wrong or duplicate header the table is invalid for TAPAS here; with a plausible but wrong header, questions that name a column select the wrong one.

</details>

In [ ]:
for item in canonical_tables:
    print({
        "table":item["source_id"],
        "detection_iou":next(r["best_iou"] for r in det_rows if r["table_id"]==item["source_id"]),
        "gold_crop_valid":item["gold_recon"]["valid"],
        "detected_crop_valid":item["det_recon"]["valid"],
        "gold_shape":[item["n_rows"],item["n_cols"]],
        "gold_crop_pred_shape":[len(item["gold_grid_pred"]["rows"]),len(item["gold_grid_pred"]["columns"])],
        "det_crop_pred_shape":[len(item["det_grid_pred"]["rows"]),len(item["det_grid_pred"]["columns"])],
        "cell_accuracy_gold_crop":item["gold_recon"].get("assignment_accuracy"),
        "cell_accuracy_detected_crop":item["det_recon"].get("assignment_accuracy"),
    })

## 25. Resource comparison

> **Evaluation practice**

Models are loaded sequentially. The table records model size, load time, stage latency and peak accelerator memory where available.

In [ ]:
def median(x):return float(np.median(x)) if x else None
resource_rows=[
    {
        "stage":"detection","model":"Table Transformer Detection",
        "model_id":DETECTION_MANIFEST["modelId"],"revision":DETECTION_MANIFEST["revision"],
        "parameter_count":det_params,
        "weight_bytes":next(x["bytes"] for x in DETECTION_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "load_seconds":det_load_seconds,"mean_latency_seconds":float(np.mean(det_times)),
        "median_latency_seconds":median(det_times),"peak_gpu_memory_bytes":det_peak,
    },
    {
        "stage":"structure","model":"Table Transformer Structure",
        "model_id":STRUCTURE_MANIFEST["modelId"],"revision":STRUCTURE_MANIFEST["revision"],
        "parameter_count":struct_params,
        "weight_bytes":next(x["bytes"] for x in STRUCTURE_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "load_seconds":struct_load_seconds,"mean_latency_seconds":float(np.mean(struct_times)),
        "median_latency_seconds":median(struct_times),"peak_gpu_memory_bytes":struct_peak,
    },
    {
        "stage":"qa","model":"TAPAS Large WTQ",
        "model_id":TAPAS_MANIFEST["modelId"],"revision":TAPAS_MANIFEST["revision"],
        "parameter_count":tapas_params,
        "weight_bytes":next(x["bytes"] for x in TAPAS_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "load_seconds":tapas_load_seconds,
        "mean_latency_seconds":float(np.mean(gold_tapas_times+structure_tapas_times+end_tapas_times)),
        "median_latency_seconds":median(gold_tapas_times+structure_tapas_times+end_tapas_times),
        "peak_gpu_memory_bytes":torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None,
    },
]
for row in resource_rows:print(row)

## 26. Qualitative four-stage panels

> **Evaluation practice**

Each canonical table gets a panel in `examples/` showing:

1. synthetic document page + predicted table box;
2. detected crop + row/column structure;
3. reconstructed table text;
4. one representative TAPAS result.

Visualizations supplement machine-readable outputs.

In [ ]:
examples_dir=Path(OUTPUT_DIR)/"examples"
examples_dir.mkdir(parents=True,exist_ok=True)

BOX_COLORS={"table":"red","table row":"royalblue","table column":"seagreen","table spanning cell":"magenta"}

def draw_boxes(image,items):
    out=image.convert("RGB").copy()
    d=ImageDraw.Draw(out)
    for item in items:
        # PIL draws no outline unless a colour is given.
        d.rectangle(item["box"],outline=BOX_COLORS.get(item["label"],"orange"),width=2)
        if item["label"]=="table":
            d.text((item["box"][0]+4,item["box"][1]-14),f"table {item.get('score',0):.2f}",fill="red")
    return out

for item in canonical_tables:
    page=draw_boxes(item["page"],item["detections"][:3])
    crop=item["det_crop"] if item["det_crop"] is not None else item["gold_crop"]
    struct=item["det_structure_raw"] if item["det_crop"] is not None else item["gold_structure_raw"]
    crop_vis=draw_boxes(crop,[x for x in struct if x["label"] in ("table row","table column","table spanning cell")])
    page.thumbnail((700,700)); crop_vis.thumbnail((700,700))
    recon=item["det_recon"] if item["det_crop"] is not None else item["gold_recon"]
    recon_lines=[" | ".join(row) for row in (recon["table"] or [])[:3]] or [f"(no table: {recon['reason']})"]
    canvas=Image.new("RGB",(max(page.width,crop_vis.width),page.height+crop_vis.height+120+30*len(recon_lines)),"white")
    canvas.paste(page,(0,0)); canvas.paste(crop_vis,(0,page.height))
    q=next(r for r in end_to_end_qa if r["table_id"]==item["source_id"])
    d=ImageDraw.Draw(canvas)
    y=page.height+crop_vis.height+8
    for line in recon_lines:
        d.text((5,y),f"Reconstructed: {line[:110]}",fill="black"); y+=30
    d.text((5,y),f"Q: {q['question']}",fill="black")
    d.text((5,y+30),f"Gold: {q['gold_denotation']}",fill="black")
    d.text((5,y+60),f"Path status: {q['pipeline_status']}  TAPAS: {q['predicted_denotation']}  aggregation: {q['predicted_aggregation']}  cells: {q['selected_cells'][:4]}",fill="black")
    path=examples_dir/f"{item['source_id'].replace('/','_')}.png"
    canvas.save(path)
    print(path)

## 27. Machine-readable exports

> **Infrastructure** — environment, pinned downloads, integrity checks and data plumbing. You may run the next cell without studying its implementation; its code is collapsed where your notebook viewer supports it.

Outputs include stage metrics, reconstructed tables, QA records, waterfall, resource metrics and provenance.

In [ ]:
import csv
from datetime import datetime, timezone

out_dir=Path(OUTPUT_DIR)
out_dir.mkdir(parents=True,exist_ok=True)
tables_dir=out_dir/"tables"
tables_dir.mkdir(exist_ok=True)

def write_csv(path,rows,fieldnames):
    with open(path,"w",encoding="utf-8",newline="") as f:
        w=csv.DictWriter(f,fieldnames=fieldnames); w.writeheader()
        for row in rows:
            out={}
            for k in fieldnames:
                v=row.get(k)
                if isinstance(v,(list,dict)):v=json.dumps(v)
                out[k]=v
            w.writerow(out)

write_csv(out_dir/"table_detection.csv",det_rows,["table_id","predicted","best_iou","hit50","hit75"])
write_csv(
    out_dir/"structure_metrics.csv",structure_rows,
    ["table_id","path","gold_rows","predicted_rows","gold_columns","predicted_columns",
     "row_count_exact","column_count_exact","grid_shape_exact","mean_row_iou","mean_column_iou"]
)

recon_rows=[]
for item in canonical_tables:
    for path_name,key in [("gold_crop","gold_recon"),("detected_crop","det_recon")]:
        r=item[key]
        recon_rows.append({
            "table_id":item["source_id"],"path":path_name,
            "gold_rows":item["n_rows"],"gold_columns":item["n_cols"],
            "predicted_rows":len(item["gold_grid_pred"]["rows"] if path_name=="gold_crop" else item["det_grid_pred"]["rows"]),
            "predicted_columns":len(item["gold_grid_pred"]["columns"] if path_name=="gold_crop" else item["det_grid_pred"]["columns"]),
            "mapped_cells":r.get("mapped",0),"unmapped_cells":r.get("unmapped",0),
            "ambiguous_cells":r.get("ambiguous",0),"cell_assignment_accuracy":r.get("assignment_accuracy"),
            "header_exact":r.get("headers_valid",False),"valid_for_tapas":r.get("valid",False),
        })
        if r.get("table") is not None:
            (tables_dir/f"{item['source_id'].replace('/','_')}_{path_name}.json").write_text(
                json.dumps({"table":r["table"],"findings":r["reason"]},indent=2),encoding="utf-8"
            )
    (tables_dir/f"{item['source_id'].replace('/','_')}_gold.json").write_text(
        json.dumps({"table":item["grid"]},indent=2),encoding="utf-8"
    )
write_csv(
    out_dir/"reconstruction_metrics.csv",recon_rows,
    ["table_id","path","gold_rows","gold_columns","predicted_rows","predicted_columns","mapped_cells",
     "unmapped_cells","ambiguous_cells","cell_assignment_accuracy","header_exact","valid_for_tapas"]
)

with open(out_dir/"qa_questions.jsonl","w",encoding="utf-8") as f:
    for r in qa_records:f.write(json.dumps(r,ensure_ascii=False)+"\n")

qa_all=gold_qa+structure_qa+end_to_end_qa
write_csv(
    out_dir/"qa_predictions.csv",qa_all,
    ["question_id","table_id","path","question","gold_aggregation","predicted_aggregation",
     "gold_denotation","predicted_denotation","denotation_correct","aggregation_correct",
     "selected_cells","coordinates","pipeline_status","latency_seconds"]
)
write_csv(
    out_dir/"resource_metrics.csv",resource_rows,
    ["stage","model","model_id","revision","parameter_count","weight_bytes","load_seconds",
     "mean_latency_seconds","median_latency_seconds","peak_gpu_memory_bytes"]
)
(out_dir/"waterfall.json").write_text(json.dumps(waterfall,indent=2),encoding="utf-8")

input_manifest={
    "source_tables":sum(len(ds[s]) for s in ("train","test")),
    "derived_tables":len(derived),
    "paper_split":{k:{"tables":len(v),"papers":len({x["paper_id"] for x in v})} for k,v in splits.items()},
    "canonical_table_ids":[x["source_id"] for x in canonical_tables],
    "carrier_geometry_sha256":CARRIER_GEOMETRY_SHA256,
    "pages":[
        {"table_id":x["source_id"],"source_pixel_sha256":x["pixel_sha256"],
         "page_sha256":x["page_sha256"],"placement":x["page_gt_box"]}
        for x in canonical_tables
    ],
    "complex_probe_ids":[x["source_id"] for x in complex_tables],
}
(out_dir/"input_manifest.json").write_text(json.dumps(input_manifest,indent=2),encoding="utf-8")

provenance={
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "notebook_spec":"2.1","profile":"TASK-INFERENCE","pedagogical_mode":"WORKSHOP",
    "dataset":{
        "id":SCITSR_REPO,"revision":SCITSR_REVISION,"license":"CC0-1.0",
        "source_files":SCITSR_FILES,"split_seed":42,
        "derivation":"pinned DIMER carrier geometry (sha256 recorded in input_manifest.json) plus cell text matched to SciTSR text chunks",
    },
    "models":{
        "detection":{"id":DETECTION_MANIFEST["modelId"],"revision":DETECTION_MANIFEST["revision"]},
        "structure":{"id":STRUCTURE_MANIFEST["modelId"],"revision":STRUCTURE_MANIFEST["revision"]},
        "tapas":{"id":TAPAS_MANIFEST["modelId"],"revision":TAPAS_MANIFEST["revision"]},
    },
    "thresholds":{
        "detection":DETECTION_THRESHOLD,"structure":STRUCTURE_THRESHOLD,"grid_nms_iou":GRID_NMS_IOU
    },
    "runtime":RUNTIME,
}
(out_dir/"provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")

print("Outputs:")
for p in sorted(out_dir.iterdir()):print(" ",p)

## 28. BYOD (optional)

> **Optional** — skipped on the canonical `Run all` path (`USE_BYOD = False`).

A production table-intelligence workflow needs text to populate predicted cells.

BYOD therefore requires:

- page image;
- OCR/text boxes;
- optional QA questions.

This notebook does not run OCR itself.

Recommended structure:

```text
dataset/
  pages/
    page001.png
  ocr.csv
  questions.csv   # optional
```

`ocr.csv`:

```text
page_id,text,x0,y0,x1,y1,order
```

OCR words would be assigned to predicted cells by word-center containment and joined in supplied reading order.

### Privacy

Tables may contain financial, personal, employee, medical or proprietary information. Do not upload restricted or regulated documents to a hosted runtime unless authorized.

In [ ]:
import csv

byod_result=None
if USE_BYOD:
    root=Path(BYOD_PATH)
    pages_dir=root/"pages"
    ocr_path=root/"ocr.csv"
    questions_path=root/"questions.csv"
    if not pages_dir.is_dir() or not ocr_path.is_file():
        raise FileNotFoundError("BYOD_PATH must contain pages/ and ocr.csv")

    with open(ocr_path,encoding="utf-8",newline="") as f:
        ocr_rows=list(csv.DictReader(f))
    required={"page_id","text","x0","y0","x1","y1","order"}
    if not ocr_rows or not required.issubset(ocr_rows[0]):
        raise ValueError(f"ocr.csv must contain {sorted(required)}")

    ocr_by_page=defaultdict(list)
    for row in ocr_rows:
        ocr_by_page[row["page_id"]].append({
            "text":row["text"],
            "box":[float(row[k]) for k in ("x0","y0","x1","y1")],
            "order":int(row["order"]),
        })
    for page_id in ocr_by_page:
        ocr_by_page[page_id].sort(key=lambda x:x["order"])

    q_by_page=defaultdict(list)
    if questions_path.is_file():
        with open(questions_path,encoding="utf-8",newline="") as f:
            question_rows=list(csv.DictReader(f))
        if question_rows and not {"id","page_id","question"}.issubset(question_rows[0]):
            raise ValueError("questions.csv must contain id,page_id,question; accepted_answer is optional")
        for row in question_rows:
            q_by_page[row["page_id"]].append(row)

    # Reload the two vision stages sequentially for BYOD.
    dp=AutoImageProcessor.from_pretrained(str(DETECTION_DIR),local_files_only=True,trust_remote_code=False)
    dm=TableTransformerForObjectDetection.from_pretrained(
        str(DETECTION_DIR),local_files_only=True,trust_remote_code=False,use_pretrained_backbone=False
    ).to(DEVICE).eval()
    det_processor,det_model=dp,dm

    byod_pages=[]
    for image_path in sorted(pages_dir.iterdir()):
        if image_path.suffix.lower() not in {".png",".jpg",".jpeg",".webp",".tif",".tiff",".bmp"}:
            continue
        page_id=image_path.stem
        if page_id not in ocr_by_page:
            continue
        image=Image.open(image_path); image.load(); image=image.convert("RGB")
        if min(image.size)<16 or max(image.size)>4096:
            raise ValueError(f"{image_path.name}: page side outside 16..4096")
        detections=detect_table(image)
        selected=detections[0] if detections else None
        byod_pages.append({"page_id":page_id,"image":image,"detection":selected})

    del det_model,det_processor
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

    sp=AutoImageProcessor.from_pretrained(
        str(STRUCTURE_DIR),local_files_only=True,trust_remote_code=False,
        size={"shortest_edge":800,"longest_edge":800},
    )
    sm=TableTransformerForObjectDetection.from_pretrained(
        str(STRUCTURE_DIR),local_files_only=True,trust_remote_code=False,use_pretrained_backbone=False
    ).to(DEVICE).eval()
    struct_processor,struct_model=sp,sm

    byod_outputs=[]
    for page in byod_pages:
        if page["detection"] is None:
            byod_outputs.append({"page_id":page["page_id"],"status":"detection_miss","table":None,"answers":[]})
            continue
        crop,crop_box=padded_crop(page["image"],page["detection"]["box"])
        raw=recognize_structure(crop)
        grid=grid_from_structure(raw)
        if not grid["valid_geometry"] or not grid["rows"] or not grid["columns"]:
            byod_outputs.append({"page_id":page["page_id"],"status":"grid_invalid","table":None,"answers":[]})
            continue

        # Assign OCR word centers into predicted row/column cells.
        strings=[[[] for _ in grid["columns"]] for __ in grid["rows"]]
        cx0,cy0,_,_=crop_box
        for word in ocr_by_page[page["page_id"]]:
            x=(word["box"][0]+word["box"][2])/2-cx0
            y=(word["box"][1]+word["box"][3])/2-cy0
            ris=[i for i,r in enumerate(grid["rows"]) if contains(r["box"],x,y)]
            cis=[i for i,c in enumerate(grid["columns"]) if contains(c["box"],x,y)]
            if len(ris)==1 and len(cis)==1:
                strings[ris[0]][cis[0]].append((word["order"],word["text"]))
        table=[[" ".join(t for _o,t in sorted(cell)).strip() for cell in row] for row in strings]
        headers=[norm_text(x) for x in table[0]] if table else []
        valid=bool(table and all(headers) and len(set(headers))==len(headers) and len(table)<=64 and len(table[0])<=32)
        if not valid:
            byod_outputs.append({"page_id":page["page_id"],"status":"header_or_limit_invalid","table":table,"answers":[]})
            continue

        answers=[]
        for q in q_by_page.get(page["page_id"],[]):
            result=tapas_answer(table,q["question"])
            row={"id":q["id"],"question":q["question"],"answer":result["answer"],
                 "aggregation":result["aggregation"],"cells":result["cells"]}
            accepted=(q.get("accepted_answer") or "").strip()
            if accepted:
                predicted=result["numeric_answer"] if result["aggregation"]!="NONE" else ", ".join(result["cells"])
                row["accepted_answer"]=accepted
                row["exact_string_match"]=norm_cell(predicted)==norm_cell(accepted)
            answers.append(row)
        byod_outputs.append({"page_id":page["page_id"],"status":"ok","table":table,"answers":answers})

    byod_result={"pages":len(byod_outputs),"outputs":byod_outputs}
    print({"byod_pages":len(byod_outputs),"status_counts":dict(__import__("collections").Counter(x["status"] for x in byod_outputs))})
else:
    print("BYOD disabled on canonical Run all path.")

## 29. Interpretation and limitations

### Table Intelligence is a pipeline

Detection, structure, text extraction, reconstruction and QA are separable failure points.

### Geometry is not text

Table Transformer predicts boxes. A real deployment still needs PDF text extraction, OCR or another document-extraction model.

### Gold-table QA is an upper-stage diagnostic

Strong TAPAS performance on the gold logical table does not imply strong end-to-end performance.

### Structure errors alter semantics

Missing or duplicated rows/columns can turn a correct source table into a different structured table before TAPAS sees it.

### Complex tables need richer reconstruction

Spanning cells, projected headers and nested structure require more than simple row × column intersection.

### Tutorial evidence is not a benchmark

This notebook uses a small, deterministic public-domain scientific-table sample and one seeded composition policy. Results are measurements from this notebook, not production-quality claims.

## Your conclusion

Complete this template from the outputs above, citing the printed number or exported file you used for each claim. Do not generalise beyond ten scientific tables, 50 generated questions and one run.

- **Task.** This notebook composed ___ → ___ → ___ to answer questions about tables on document pages.
- **Principal result.** End-to-end denotation accuracy was ___ on 50 questions, against ___ on the gold table (the reference) and ___ on the structure-only path.
- **Where the loss entered.** The largest drop occurred between ___ and ___ (evidence: ___ in `waterfall.json` or `reconstruction_metrics.csv`), mainly because ___.
- **Failure mode or uncertainty.** One important failure was ___. With ten tables, a single table moves a per-table rate by 10 percentage points and five questions.
- **Limitations.** Cell text came from annotations rather than OCR; each synthetic page held one table; only rectangular tables were used for QA; one paper-disjoint sample and one composition policy were measured.

<details>
<summary>Sample conclusion (illustrative; your numbers may differ)</summary>

On ten held-out SciTSR-PD tables, TAPAS answered 80% of 50 questions correctly from the gold tables and from gold-crop reconstructions, but 60% end to end. Every gold crop produced an exact grid, so structure recognition on a well-framed crop was not the bottleneck here. The loss entered at detection (one page had no detection above 0.90, which made its five questions unanswerable) and on one detected crop whose grid gained a column. These are measurements on ten tables with annotation text; they do not predict accuracy on scanned documents, OCR text, or tables with spanning cells.

</details>

## Troubleshooting

| Symptom | Likely cause | What to do |
|---|---|---|
| Run is very slow; `device` prints `cpu` | no GPU runtime selected | select a T4 GPU and `Run all` again (CPU works, but TAPAS takes about a second per question) |
| `pip install failed …` or `Pinned install failed` | package index unreachable, or a preinstalled package conflicts | read the printed pip error; rerun the Runtime cell; restart the runtime if imports still report old versions |
| `… requires the timm library` | the Runtime cell was skipped | run the Runtime cell (§4) first |
| model size or SHA-256 mismatch | incomplete or changed download | delete that model's folder under `weights/` and rerun; never bypass the check |
| SciTSR-PD shard size or SHA-256 mismatch | incomplete download | delete `weights/scitsr-pd` and rerun |
| `Embedded carrier geometry does not match its SHA-256` | the geometry cell was edited | restore the cell from the published notebook |
| `carrier tables failed to derive` or `source pixels differ` | a different dataset revision was read | keep the pinned `SCITSR_REVISION`; do not relax the check |
| split or eligibility `RuntimeError` | seed, derivation or eligibility rules were edited | restore the defaults; never borrow train/validation tables for the QA set |
| CUDA out of memory | another notebook shares the GPU, or cells were rerun out of order | restart the runtime and `Run all`; models are loaded one at a time |
| every reconstruction invalid | Configuration thresholds were changed | restore the defaults and experiment in the activity cell instead |
| BYOD: missing `pages/` or `ocr.csv` | wrong folder layout | follow the layout in §28 |

## Glossary

| Term | Meaning |
|---|---|
| **Table detection** | Finding the bounding box of each table on a page |
| **Structure recognition** | Finding the rows, columns, headers and spanning cells inside a table crop |
| **Spanning cell** | A cell that covers more than one row or column |
| **Crop / coordinate frame** | A sub-image; boxes must be shifted between page, crop and table coordinates |
| **IoU** | Intersection over union of two boxes: 1.0 is identical, 0 is disjoint |
| **AP50 / AP75** | Average precision counting a detection as correct at IoU ≥ 0.50 / ≥ 0.75 |
| **NMS** | Non-maximum suppression: drop a box that overlaps a higher-scoring box of the same label |
| **Grid reconstruction** | Intersecting predicted rows and columns to form cells |
| **Cell-text assignment** | Placing each text item into the predicted cell that contains its center |
| **TAPAS** | A BERT-style model that answers questions over a table of strings by selecting cells and an aggregation |
| **WTQ** | WikiTableQuestions, the QA dataset TAPAS Large WTQ was fine-tuned on |
| **Aggregation operator** | `NONE`, `SUM`, `AVERAGE` or `COUNT` applied to the selected cells |
| **Denotation** | The final answer value (cell strings or a number), compared with the gold answer |
| **Paper-disjoint split** | Tables from one paper never appear in two splits |
| **Carrier** | The DIMER repository whose pinned model and data this notebook reproduces |
| **Error waterfall** | Stage-by-stage counts showing where end-to-end failures entered |

## 30. Terminal summary

The final cell verifies required outputs and prints the stage waterfall without declaring a winner.

In [ ]:
required=[
    out_dir/"table_detection.csv",
    out_dir/"structure_metrics.csv",
    out_dir/"reconstruction_metrics.csv",
    out_dir/"qa_questions.jsonl",
    out_dir/"qa_predictions.csv",
    out_dir/"waterfall.json",
    out_dir/"resource_metrics.csv",
    out_dir/"input_manifest.json",
    out_dir/"provenance.json",
]
missing=[str(p) for p in required if not p.is_file()]
if missing:raise RuntimeError(f"Required outputs missing: {missing}")

print("DIMER Table Intelligence Workshop")
print("-"*34)
print(f"Canonical tables: {len(canonical_tables)}")
print(f"Questions: {len(qa_records)}")
print()
print("Stage 1 — Detection")
print(f"  AP50: {det_metrics['ap50']:.3f}")
print(f"  hit@0.50: {det_metrics['hit50']:.3f}")
print(f"  mean IoU: {det_metrics['mean_best_iou']:.3f}")
print()
print("Stage 2 — Structure")
print(f"  gold-crop exact grids: {waterfall['structure_gold_crop']['grid_exact']} / {len(canonical_tables)}")
print(f"  detected-crop exact grids: {waterfall['structure_detected_crop']['grid_exact']} / {len(canonical_tables)}")
print()
print("Stage 3 — Reconstruction")
print(f"  valid gold-crop tables: {waterfall['reconstruction']['valid_gold_crop']} / {len(canonical_tables)}")
print(f"  valid detected tables: {waterfall['reconstruction']['valid_detected_crop']} / {len(canonical_tables)}")
print()
print("Stage 4 — TAPAS")
print(f"  gold-table denotation: {waterfall['qa']['gold_table_denotation_accuracy']:.3f}")
print(f"  structure-only denotation: {waterfall['qa']['structure_only_denotation_accuracy']:.3f}")
print(f"  end-to-end denotation: {waterfall['qa']['end_to_end_denotation_accuracy']:.3f}")
print()
print(f"Outputs: {out_dir}/")